# M3 다음 실험: 손실·입력 범위·근거 상태

기존 실패 실험은 보존합니다. 이 노트북만 위에서 아래로 실행하세요. 기존 Colab 노트북의 모두 실행은 사용하지 않습니다. 입력 파일 `input.tar.gz`를 Colab 세션 저장소에 업로드해야 합니다. source는 아래 셀에 포함되어 있습니다. Training 23,408 / 내부 검증 5,792, 임계값 0.5를 유지합니다. 공식 Validation은 자동으로 읽지 않습니다.

GPU 전체 실행은 누적 6시간으로 제한됩니다. 예산이 끝나면 결과를 내려받고 같은 파일·설정으로 재개하세요. 규칙으로 생성한 근거 상태는 약한 감독 신호이며 수동 주석으로 표시하지 않습니다.

In [ ]:
import os,sys,subprocess
from pathlib import Path
import json,hashlib,tarfile,zipfile
os.environ['USE_TF']='0'
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','scikit-learn','matplotlib','numpy','pandas'],check=True)
import torch
assert torch.cuda.is_available(), 'GPU 런타임이 필요합니다.'
print('Runtime:',sys.version,torch.__version__,torch.cuda.get_device_name(0))


In [ ]:
# Source-only files; contains no Training transcript or labels.
ROOT=Path('/content/m3_next')
ROOT.mkdir(exist_ok=True)
SOURCES={'src/dcc_data/__init__.py': '"""Minimal, rule-aware data access utilities for DCC 2026."""\n\nfrom .manifest import ManifestRow, build_manifest, load_manifest, write_manifest\n\n__all__ = ["ManifestRow", "build_manifest", "load_manifest", "write_manifest"]\n', 'src/dcc_data/manifest.py': '"""Build and load a location-only manifest across all download shards."""\n\nfrom __future__ import annotations\n\nimport csv\nimport os\nimport tempfile\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom .guards import (\n    DuplicateIdentifierError,\n    ManifestPairingError,\n    split_from_path,\n    validate_manifest_rows,\n)\n\n\nMANIFEST_VERSION = "1"\nMANIFEST_COLUMNS = (\n    "manifest_version",\n    "split",\n    "stem",\n    "wav_path",\n    "json_path",\n)\n\n\n@dataclass(frozen=True)\nclass ManifestRow:\n    split: str\n    stem: str\n    wav_path: str\n    json_path: str\n\n\ndef build_manifest(data_root: str | Path) -> list[ManifestRow]:\n    """Pair all WAV/JSON files globally by ``(split, stem)``.\n\n    Paths in returned rows are POSIX-style paths relative to ``data_root``.\n    Download shard boundaries are deliberately ignored during pairing.\n    """\n\n    root = Path(data_root).resolve()\n    if not root.is_dir():\n        raise NotADirectoryError(root)\n\n    groups: dict[tuple[str, str], dict[str, object]] = {}\n    for path in root.rglob("*"):\n        if not path.is_file() or path.suffix.lower() not in {".wav", ".json"}:\n            continue\n        relative = path.relative_to(root)\n        split = split_from_path(relative)\n        key = (split, path.stem.casefold())\n        group = groups.setdefault(\n            key,\n            {"stem": path.stem, "wav": [], "json": []},\n        )\n        role = path.suffix.lower().lstrip(".")\n        role_paths = group[role]\n        assert isinstance(role_paths, list)\n        role_paths.append(relative.as_posix())\n\n    rows: list[ManifestRow] = []\n    for (split, _), group in sorted(groups.items()):\n        wav_paths = group["wav"]\n        json_paths = group["json"]\n        assert isinstance(wav_paths, list) and isinstance(json_paths, list)\n        if len(wav_paths) > 1 or len(json_paths) > 1:\n            raise DuplicateIdentifierError(\n                f"Duplicate files for {(split, group[\'stem\'])}: "\n                f"wav={wav_paths}, json={json_paths}"\n            )\n        if len(wav_paths) != 1 or len(json_paths) != 1:\n            raise ManifestPairingError(\n                f"Unpaired files for {(split, group[\'stem\'])}: "\n                f"wav={wav_paths}, json={json_paths}"\n            )\n        rows.append(\n            ManifestRow(\n                split=split,\n                stem=str(group["stem"]),\n                wav_path=wav_paths[0],\n                json_path=json_paths[0],\n            )\n        )\n\n    validate_manifest_rows(rows, root)\n    return rows\n\n\ndef write_manifest(\n    rows: list[ManifestRow], output_path: str | Path, *, overwrite: bool = False\n) -> Path:\n    """Write a deterministic CSV without labels or transcript content."""\n\n    output = Path(output_path)\n    if output.exists() and not overwrite:\n        raise FileExistsError(f"Refusing to overwrite existing manifest: {output}")\n    output.parent.mkdir(parents=True, exist_ok=True)\n\n    temporary_name: str | None = None\n    try:\n        with tempfile.NamedTemporaryFile(\n            mode="w",\n            encoding="utf-8",\n            newline="",\n            prefix=f".{output.name}.",\n            suffix=".tmp",\n            dir=output.parent,\n            delete=False,\n        ) as stream:\n            temporary_name = stream.name\n            writer = csv.DictWriter(stream, fieldnames=MANIFEST_COLUMNS)\n            writer.writeheader()\n            for row in rows:\n                writer.writerow(\n                    {\n                        "manifest_version": MANIFEST_VERSION,\n                        "split": row.split,\n                        "stem": row.stem,\n                        "wav_path": row.wav_path,\n                        "json_path": row.json_path,\n                    }\n                )\n        os.replace(temporary_name, output)\n    except Exception:\n        if temporary_name is not None:\n            Path(temporary_name).unlink(missing_ok=True)\n        raise\n    return output\n\n\ndef load_manifest(\n    manifest_path: str | Path,\n    *,\n    data_root: str | Path | None = None,\n    validate_paths: bool = True,\n) -> list[ManifestRow]:\n    """Load a manifest and re-run integrity guards."""\n\n    path = Path(manifest_path)\n    rows: list[ManifestRow] = []\n    with path.open("r", encoding="utf-8", newline="") as stream:\n        reader = csv.DictReader(stream)\n        if tuple(reader.fieldnames or ()) != MANIFEST_COLUMNS:\n            raise ValueError(\n                f"Unexpected manifest columns: {reader.fieldnames}; "\n                f"expected {MANIFEST_COLUMNS}"\n            )\n        for line_number, item in enumerate(reader, start=2):\n            if item["manifest_version"] != MANIFEST_VERSION:\n                raise ValueError(\n                    f"Unsupported manifest version on line {line_number}: "\n                    f"{item[\'manifest_version\']!r}"\n                )\n            rows.append(\n                ManifestRow(\n                    split=item["split"],\n                    stem=item["stem"],\n                    wav_path=item["wav_path"],\n                    json_path=item["json_path"],\n                )\n            )\n\n    validation_root = data_root if validate_paths else None\n    if validate_paths and validation_root is None:\n        raise ValueError("data_root is required when validate_paths=True")\n    validate_manifest_rows(rows, validation_root)\n    return rows\n', 'src/dcc_data/guards.py': '"""Fail-fast guards for manifest integrity and split isolation."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable, Mapping\nfrom pathlib import Path\nfrom typing import Any\n\n\nTRAINING = "Training"\nVALIDATION = "Validation"\nVALID_SPLITS = frozenset({TRAINING, VALIDATION})\n\n\nclass DataGuardError(ValueError):\n    """Base class for DCC data guard failures."""\n\n\nclass UnknownSplitError(DataGuardError):\n    """Raised when a path does not contain exactly one recognized split."""\n\n\nclass ManifestPairingError(DataGuardError):\n    """Raised when a stem does not have exactly one WAV and one JSON file."""\n\n\nclass DuplicateIdentifierError(DataGuardError):\n    """Raised when a split contains a duplicate identifier."""\n\n\nclass CrossSplitIdentifierError(DataGuardError):\n    """Raised when an identifier appears in both Training and Validation."""\n\n\nclass SplitContaminationError(DataGuardError):\n    """Raised when data from an unexpected split reaches a guarded path."""\n\n\ndef split_from_path(path: str | Path) -> str:\n    """Return the exact split component from a relative or absolute path."""\n\n    candidates = [part for part in Path(path).parts if part in VALID_SPLITS]\n    if len(candidates) != 1:\n        raise UnknownSplitError(\n            f"Expected exactly one Training/Validation path component, got "\n            f"{candidates!r}: {path}"\n        )\n    return candidates[0]\n\n\ndef validate_manifest_rows(\n    rows: Iterable[Any], data_root: str | Path | None = None\n) -> None:\n    """Validate uniqueness, path/split agreement, and optional file existence."""\n\n    materialized = list(rows)\n    root = Path(data_root).resolve() if data_root is not None else None\n    seen: set[tuple[str, str]] = set()\n    stems_by_split = {TRAINING: set(), VALIDATION: set()}\n\n    for row in materialized:\n        if row.split not in VALID_SPLITS:\n            raise UnknownSplitError(f"Unknown manifest split: {row.split!r}")\n\n        key = (row.split, row.stem.casefold())\n        if key in seen:\n            raise DuplicateIdentifierError(f"Duplicate manifest key: {key}")\n        seen.add(key)\n        stems_by_split[row.split].add(row.stem.casefold())\n\n        for role, relative_path, suffix in (\n            ("wav", row.wav_path, ".wav"),\n            ("json", row.json_path, ".json"),\n        ):\n            path = Path(relative_path)\n            if path.suffix.lower() != suffix:\n                raise ManifestPairingError(\n                    f"{role} path has the wrong extension for {key}: {relative_path}"\n                )\n            actual_split = split_from_path(path)\n            if actual_split != row.split:\n                raise SplitContaminationError(\n                    f"Manifest split {row.split!r} disagrees with {role} path "\n                    f"split {actual_split!r}: {relative_path}"\n                )\n            if root is not None and not (root / path).is_file():\n                raise FileNotFoundError(root / path)\n\n    overlap = stems_by_split[TRAINING] & stems_by_split[VALIDATION]\n    if overlap:\n        preview = sorted(overlap)[:10]\n        raise CrossSplitIdentifierError(\n            f"{len(overlap)} stem(s) occur in both splits; examples: {preview}"\n        )\n\n\ndef ensure_split(rows: Iterable[Any], expected_split: str) -> list[Any]:\n    """Materialize rows and fail if any row is from another split."""\n\n    if expected_split not in VALID_SPLITS:\n        raise UnknownSplitError(f"Unknown expected split: {expected_split!r}")\n    materialized = list(rows)\n    wrong = [row for row in materialized if row.split != expected_split]\n    if wrong:\n        examples = [(row.split, row.stem) for row in wrong[:10]]\n        raise SplitContaminationError(\n            f"Expected only {expected_split}, found {len(wrong)} foreign row(s): "\n            f"{examples}"\n        )\n    return materialized\n\n\ndef ensure_training_rows(rows: Iterable[Any]) -> list[Any]:\n    """Return rows only when every row belongs to Training."""\n\n    return ensure_split(rows, TRAINING)\n\n\ndef ensure_training_batch(batch: Mapping[str, Any]) -> None:\n    """Fail immediately if a batch contains a non-Training provenance split."""\n\n    if "split" not in batch:\n        raise SplitContaminationError("Training batch has no split provenance")\n    value = batch["split"]\n    splits = [value] if isinstance(value, str) else list(value)\n    foreign = [split for split in splits if split != TRAINING]\n    if foreign:\n        raise SplitContaminationError(\n            f"Training batch contains non-Training split(s): {foreign[:10]}"\n        )\n', 'src/dcc_inference/__init__.py': '"""Per-mission inference modules behind the root ``inference.py``."""', 'src/dcc_data/mission3_context_experiment.py': '"""Reproducible M3 context experiment; official validation is a separate command.\n\nNo official labels enter preparation, retrieval, head fitting, or encoder fitting.\nThe exact existing holdout is recovered by file name from saved baseline arrays.\nFeature shards and completed runs resume only after source/config hash checks.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport hashlib\nimport json\nimport random\nimport re\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import f1_score, precision_score, recall_score\nfrom torch.nn import functional as F\n\nfrom dcc_inference import mission3_bert as baseline\nfrom dcc_inference import mission3_context as context\n\nRECIPE = dict(head_epochs=10, head_lr=.001, head_batch=128, weight_decay=.01,\n              fine_epochs=1, encoder_lr=.000001, fine_head_lr=.0001,\n              fine_batch=16, micro_batch=8, bootstrap=1000, seeds=[42, 43, 44])\nKEYWORDS = {\n    "고열": r"열이|고열|열\\s*나|\\d{2}\\.\\d\\s*도|체온",\n    "구토": r"토했|토해|토하|토를|구토|게워|토가|오바이트",\n    "두통": r"머리.{0,8}(아프|아파|아퍼|깨질|지끈|띵|쑤)|두통|편두통",\n    "복통": r"배.{0,6}(아프|아파|아퍼)|복통|배가|명치|아랫배|윗배",\n    "어지러움": r"어지러|어지럽|어질|빙빙|핑\\s*돌|현기증",\n    "열상": r"찢어|베였|베이|피가|피\\s*나|출혈|꿰매|상처",\n    "오심": r"메스껍|메슥|울렁|구역|토할\\s*것|토할\\s*거|속이\\s*(안\\s*좋|이상|불편|뒤집)|오심",\n    "전신쇠약": r"기운이?\\s*없|힘이?\\s*없|쳐지|처지|축\\s*늘어|기력|무기력|못\\s*일어|쇠약",\n    "호흡곤란": r"숨.{0,6}(못|안|차|막|쉬기|가빠|헐떡)|호흡|숨이",\n}\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1 << 20), b""):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef save_torch(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef read_rows(label_dir):\n    paths = sorted(Path(label_dir).rglob("*.json"))\n    if not paths or {p.casefold() for p in Path(label_dir).parts} & {"validation", "test"}:\n        raise ValueError("A nonempty Training label directory is required")\n    rows = []\n    for path in paths:\n        doc = json.loads(path.read_text(encoding="utf-8-sig"))\n        text, spans = context.text_and_spans(doc)\n        labels = doc.get("symptom", [])\n        if isinstance(labels, str):\n            labels = [labels]\n        labels = {str(s).strip() for s in labels}\n        # Strip every non-text input field before caching or encoding.\n        allowed = {"utterances": [{"text": text[start:end]} for start, end in spans]}\n        rows.append(dict(name=path.name, call=context.call_key(path.name), text_hash=context.text_hash(text),\n                         doc=allowed, gold=[int(s in labels) for s in context.LABELS]))\n    if len({row["name"] for row in rows}) != len(rows):\n        raise ValueError("Duplicate file names; cannot align with baseline predictions")\n    return rows\n\n\ndef prepare_data(label_dir, reference_dir, smoke_rows=0):\n    rows = read_rows(label_dir)\n    references = {}\n    for name in ("r3", "k12", "p0"):\n        with np.load(Path(reference_dir) / f"{name}_holdout.npz", allow_pickle=False) as archive:\n            references[name] = {key:archive[key].copy() for key in ("names", "gold", "probs")}\n    holdout_names = references["r3"]["names"].astype(str).tolist()\n    if len(set(holdout_names)) != len(holdout_names):\n        raise ValueError("Duplicate baseline holdout names")\n    lookup = {row["name"]:i for i,row in enumerate(rows)}\n    if not set(holdout_names) <= set(lookup):\n        raise ValueError("Baseline holdout files missing from Training")\n    holdout = [lookup[name] for name in holdout_names]\n    holdout_set = set(holdout_names)\n    train = [i for i in range(len(rows)) if rows[i]["name"] not in holdout_set]\n    expected_gold = np.asarray([rows[i]["gold"] for i in holdout])\n    aligned = {}\n    for name, arrays in references.items():\n        indices = {str(file):i for i,file in enumerate(arrays["names"])}\n        if len(indices) != len(holdout) or set(indices) != set(holdout_names):\n            raise ValueError(f"{name}: holdout file set differs")\n        order = [indices[file] for file in holdout_names]\n        if not np.array_equal(expected_gold, arrays["gold"][order]):\n            raise ValueError(f"{name}: baseline gold differs from provided labels")\n        aligned[name] = arrays["probs"][order]\n    ensemble = .45*aligned["r3"] + .15*aligned["k12"] + .40*aligned["p0"]\n    if not smoke_rows and (len(train), len(holdout)) != (23408, 5792):\n        raise ValueError(f"Expected existing 23408/5792 split; got {len(train)}/{len(holdout)}")\n    if {rows[i]["call"] for i in train} & {rows[i]["call"] for i in holdout}:\n        raise ValueError("Call identity overlaps between Training and holdout")\n    if smoke_rows:\n        train = train[:smoke_rows]\n        selected = list(range(min(max(8, smoke_rows//4), len(holdout))))\n        holdout = [holdout[i] for i in selected]\n        ensemble = ensemble[selected]\n        aligned = {name:values[selected] for name,values in aligned.items()}\n    return rows, train, holdout, aligned["r3"], ensemble\n\n\nclass BudgetReached(RuntimeError):\n    """A bounded run can resume its completed shards and epochs later."""\n\n\nclass RunBudget:\n    def __init__(self, output, hours):\n        if not 0 < hours <= 10:\n            raise ValueError("The approved run budget must be between zero and ten hours")\n        self.path = Path(output)/"budget.json"\n        self.limit = hours*3600\n        previous = json.loads(self.path.read_text(encoding="utf-8")) if self.path.exists() else {}\n        self.used = previous.get("used_seconds", 0.)\n        self.started = self.saved = time.monotonic()\n\n    def check(self, force=False):\n        now = time.monotonic()\n        used = self.used + now-self.started\n        if force or now-self.saved >= 60 or used >= self.limit:\n            write_json(self.path,dict(used_seconds=used,limit_seconds=self.limit))\n            self.saved = now\n        if used >= self.limit:\n            raise BudgetReached("Run budget reached; completed shards/epochs are resumable")\n\n\ndef check_budget(budget):\n    if budget is not None:\n        budget.check()\n\n\ndef metrics(gold, probabilities, documents=None):\n    gold, probabilities = np.asarray(gold), np.asarray(probabilities)\n    if gold.shape != probabilities.shape or gold.ndim != 2 or gold.shape[1] != 9:\n        raise ValueError("Metric arrays must align as (calls, 9)")\n    if not len(gold) or not np.isfinite(probabilities).all():\n        raise ValueError("Empty/nonfinite metric arrays")\n    prediction = probabilities >= .5\n    result = dict(rows=len(gold), macro_f1=float(f1_score(gold, prediction, average="macro", zero_division=0)),\n                  micro_f1=float(f1_score(gold, prediction, average="micro", zero_division=0)),\n                  empty_predictions=int((prediction.sum(1)==0).sum()),\n                  per_label={})\n    p = precision_score(gold, prediction, average=None, zero_division=0)\n    r = recall_score(gold, prediction, average=None, zero_division=0)\n    f = f1_score(gold, prediction, average=None, zero_division=0)\n    for i,label in enumerate(context.LABELS):\n        info = dict(f1=float(f[i]), precision=float(p[i]), recall=float(r[i]), support=int(gold[:, i].sum()))\n        if documents is not None:\n            texts = [context.text_and_spans(doc)[0] for doc in documents]\n            match = np.array([bool(re.search(KEYWORDS[label], text)) for text in texts])\n            for key, group in (("regex_match", match), ("regex_no_match", ~match)):\n                positive = group & (gold[:, i]==1)\n                info[key] = dict(rows=int(group.sum()), positives=int(positive.sum()),\n                                 recall=float(prediction[positive, i].mean()) if positive.any() else None)\n            question, negation = [], []\n            for doc in documents:\n                utterances = [u["text"] for u in doc["utterances"]]\n                pairs = [(u, utterances[j+1]) for j,u in enumerate(utterances[:-1])\n                         if re.search(KEYWORDS[label], u) and re.search(r"\\?|세요|나요|습니까|시죠|는지",u)]\n                question.append(bool(pairs))\n                negation.append(any(re.search(r"없|아니|안\\s|않", answer) for _,answer in pairs))\n            for key, mask in (("question_answer", question), ("negative_answer", negation)):\n                mask = np.asarray(mask, dtype=bool)\n                info[key] = dict(rows=int(mask.sum()), false_positive=int(((gold[:,i]==0)&prediction[:,i]&mask).sum()),\n                                 f1=float(f1_score(gold[mask,i], prediction[mask,i], zero_division=0)) if mask.any() else None)\n        result["per_label"][label] = info\n    return result\n\n\ndef loss(logits, targets, pos_weight):\n    logits, targets = logits.float(), targets.float()\n    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)\n    probability = logits.sigmoid()\n    f1 = 2*(probability*targets).sum(0)/(probability.sum(0)+targets.sum(0)+1e-8)\n    return bce + 1-f1.mean()\n\n\ndef positive_weights(rows, train, device):\n    gold = torch.tensor([rows[i]["gold"] for i in train], dtype=torch.float32, device=device)\n    positive = gold.sum(0)\n    return (.5*((len(train)-positive)/positive.clamp_min(1)).sqrt()).clamp(1,6)\n\n\ndef initialize(checkpoint, method, device, bank=None, config=None):\n    from transformers import AutoTokenizer\n    import tempfile\n    payload = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    if payload.get("format") != baseline.FORMAT or tuple(payload.get("labels",())) != context.LABELS:\n        raise ValueError("Expected an exported ROBERTA-003 baseline checkpoint")\n    base = baseline._build(payload["files"], payload)\n    base.load_state_dict(payload.pop("model"), strict=True)\n    with tempfile.TemporaryDirectory() as folder:\n        for name,text in payload["files"].items():\n            (Path(folder)/Path(name).name).write_text(text,encoding="utf-8")\n        tokenizer = AutoTokenizer.from_pretrained(folder, local_files_only=True, use_fast=True)\n    model = context.ContextModel(base, method, config, bank).to(device)\n    return model, tokenizer, payload\n\n\ndef extract_features(model, tokenizer, rows, indices, directory, signature, batch_size=32, budget=None):\n    """Resumable bounded shards; retained features contain no transcript strings."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    marker = directory / "index.json"\n    spec = dict(signature=signature, indices=indices, batch_size=batch_size)\n    if marker.exists() and json.loads(marker.read_text(encoding="utf-8")) != spec:\n        raise ValueError("Feature cache source/split/config changed")\n    write_json(marker,spec)\n    model.eval()\n    device = next(model.parameters()).device\n    features = {}\n    with torch.inference_mode():\n        for start in range(0,len(indices),batch_size):\n            check_budget(budget)\n            path = directory / f"shard_{start:06d}.pt"\n            chunk = indices[start:start+batch_size]\n            if path.exists():\n                records = torch.load(path,map_location="cpu",weights_only=True)\n                if [record["index"] for record in records] != chunk:\n                    raise ValueError("Feature shard row order changed")\n            else:\n                encoded,_ = context.tokenize_documents(tokenizer,[rows[i]["doc"] for i in chunk])\n                ids,mask,turns = [encoded[key].to(device) for key in ("input_ids","attention_mask","turn_ids")]\n                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=="cuda"):\n                    cls,logits,pooled,visible = model.extract(ids,mask,turns)\n                records = [dict(index=i,cls=cls[j].half().cpu(),base_logits=logits[j].cpu(),\n                                turns=pooled[j,visible[j]].half().cpu()) for j,i in enumerate(chunk)]\n                save_torch(path,records)\n            for record in records:\n                features[record["index"]] = record\n            if start % (batch_size*20)==0:\n                print(f"feature cache {min(start+batch_size,len(indices))}/{len(indices)}",flush=True)\n    return features\n\n\ndef make_bank(features,rows,train):\n    return context.MemoryBank(torch.stack([features[i]["cls"] for i in train]),\n                              [rows[i]["gold"] for i in train], [rows[i]["call"] for i in train],\n                              [rows[i]["text_hash"] for i in train])\n\n\ndef feature_batch(features,rows,indices,device):\n    cls = torch.stack([features[i]["cls"] for i in indices]).float().to(device)\n    logits = torch.stack([features[i]["base_logits"] for i in indices]).float().to(device)\n    length = max(1,max(len(features[i]["turns"]) for i in indices))\n    turns = torch.zeros(len(indices),length,cls.shape[-1],device=device)\n    mask = torch.zeros(len(indices),length,dtype=torch.bool,device=device)\n    for j,i in enumerate(indices):\n        n = len(features[i]["turns"])\n        turns[j,:n] = features[i]["turns"].float().to(device)\n        mask[j,:n] = True\n    return cls,logits,turns,mask\n\n\ndef evaluate_features(head,bank,features,rows,holdout,device,batch_size=128):\n    head.eval()\n    outputs = []\n    with torch.no_grad():\n        for start in range(0,len(holdout),batch_size):\n            chunk = holdout[start:start+batch_size]\n            logits = head(*feature_batch(features,rows,chunk,device),bank=bank,\n                          calls=[rows[i]["call"] for i in chunk],hashes=[rows[i]["text_hash"] for i in chunk])\n            outputs.append(logits.sigmoid().cpu().numpy())\n    return np.concatenate(outputs)\n\n\ndef fit_head(model,features,rows,train,holdout,seed,directory,signature,recipe,budget=None):\n    directory = Path(directory)\n    directory.mkdir(parents=True,exist_ok=True)\n    state_path = directory / "head_last.pt"\n    specification = dict(signature=signature,method=model.head.method,seed=seed,recipe=recipe)\n    device = next(model.head.parameters()).device\n    for parameter in model.base.parameters():\n        parameter.requires_grad_(False)\n    model.base.eval()\n    optimizer = torch.optim.AdamW(model.head.parameters(),lr=recipe["head_lr"],weight_decay=recipe["weight_decay"])\n    weights = positive_weights(rows,train,device)\n    start,history = 0,[]\n    if state_path.exists():\n        state = torch.load(state_path,map_location=device,weights_only=True)\n        if state["specification"] != specification:\n            raise ValueError("Head resume source/config mismatch")\n        model.head.load_state_dict(state["head"])\n        optimizer.load_state_dict(state["optimizer"])\n        start,history = state["epoch"],state["history"]\n    for epoch in range(start+1,recipe["head_epochs"]+1):\n        seed_all(seed+epoch)  # epoch-boundary resume has the same order and dropout RNG\n        model.head.train()\n        order = np.random.default_rng(seed+epoch).permutation(train).tolist()\n        total,started = 0.,time.monotonic()\n        for offset in range(0,len(order),recipe["head_batch"]):\n            check_budget(budget)\n            chunk = order[offset:offset+recipe["head_batch"]]\n            optimizer.zero_grad(set_to_none=True)\n            logits = model.head(*feature_batch(features,rows,chunk,device),bank=model.bank,\n                                calls=[rows[i]["call"] for i in chunk],hashes=[rows[i]["text_hash"] for i in chunk])\n            targets = torch.tensor([rows[i]["gold"] for i in chunk],device=device,dtype=torch.float32)\n            value = loss(logits,targets,weights)\n            if not torch.isfinite(value):\n                raise RuntimeError("Nonfinite head loss")\n            value.backward()\n            optimizer.step()\n            total += value.item()*len(chunk)\n        record = dict(epoch=epoch,loss=total/len(train),seconds=time.monotonic()-started)\n        if holdout:\n            probs = evaluate_features(model.head,model.bank,features,rows,holdout,device)\n            record.update(metrics([rows[i]["gold"] for i in holdout],probs))\n        history.append(record)\n        save_torch(state_path,dict(specification=specification,head=model.head.state_dict(),optimizer=optimizer.state_dict(),\n                                   epoch=epoch,history=history))\n        write_json(directory/"history.json",history)\n        print(f"{model.head.method} seed={seed} epoch={epoch} loss={record[\'loss\']:.4f} macro={record.get(\'macro_f1\')}",flush=True)\n    probabilities = evaluate_features(model.head,model.bank,features,rows,holdout,device) if holdout else None\n    if probabilities is not None:\n        np.savez_compressed(directory/"holdout.npz",names=[rows[i]["name"] for i in holdout],\n                            gold=[rows[i]["gold"] for i in holdout],probs=probabilities)\n    return probabilities\n\n\ndef fit_encoder(model,tokenizer,rows,train,holdout,seed,directory,signature,recipe,budget=None):\n    directory = Path(directory)\n    state_path = directory/"encoder_last.pt"\n    specification = dict(signature=signature,seed=seed,recipe=recipe,method=model.head.method)\n    device = next(model.parameters()).device\n    if state_path.exists():\n        state = torch.load(state_path,map_location=device,weights_only=True)\n        if state["specification"] != specification:\n            raise ValueError("Encoder resume configuration changed")\n        model.load_state_dict(state["model"])\n    else:\n        seed_all(seed)\n        for parameter in model.base.parameters():\n            parameter.requires_grad_(True)\n        optimizer = torch.optim.AdamW([dict(params=model.base.parameters(),lr=recipe["encoder_lr"]),\n                                      dict(params=model.head.parameters(),lr=recipe["fine_head_lr"])],\n                                     weight_decay=recipe["weight_decay"])\n        from transformers import get_linear_schedule_with_warmup\n        updates = math_ceil(len(train)/recipe["fine_batch"])\n        scheduler = get_linear_schedule_with_warmup(optimizer,math_ceil(updates*.1),updates)\n        scaler = torch.amp.GradScaler("cuda",enabled=device.type=="cuda")\n        weights = positive_weights(rows,train,device)\n        order = np.random.default_rng(seed).permutation(train).tolist()\n        model.train()\n        total,started = 0.,time.monotonic()\n        for offset in range(0,len(order),recipe["fine_batch"]):\n            check_budget(budget)\n            group = order[offset:offset+recipe["fine_batch"]]\n            optimizer.zero_grad(set_to_none=True)\n            for micro in range(0,len(group),recipe["micro_batch"]):\n                chunk = group[micro:micro+recipe["micro_batch"]]\n                encoded,hashes = context.tokenize_documents(tokenizer,[rows[i]["doc"] for i in chunk])\n                ids,mask,turns = [encoded[key].to(device) for key in ("input_ids","attention_mask","turn_ids")]\n                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=="cuda"):\n                    logits = model(ids,mask,turns,[rows[i]["call"] for i in chunk],hashes)\n                targets = torch.tensor([rows[i]["gold"] for i in chunk],device=device,dtype=torch.float32)\n                value = loss(logits,targets,weights)\n                if not torch.isfinite(value):\n                    raise RuntimeError("Nonfinite encoder loss")\n                scaler.scale(value*len(chunk)/len(group)).backward()\n                total += value.item()*len(chunk)\n            previous = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            if scaler.get_scale() >= previous:\n                scheduler.step()\n            if offset % (recipe["fine_batch"]*100)==0:\n                print(f"encoder {model.head.method} seed={seed} {offset}/{len(order)}",flush=True)\n        save_torch(state_path,dict(specification=specification,model=model.state_dict(),epoch=1,\n                                   loss=total/len(train),seconds=time.monotonic()-started))\n    model.eval()\n    probs = context.probabilities(model,tokenizer,[rows[i]["doc"] for i in holdout],\n                                  [rows[i]["name"] for i in holdout],batch_size=recipe["micro_batch"]) if holdout else None\n    if probs is not None:\n        np.savez_compressed(directory/"fine_holdout.npz",names=[rows[i]["name"] for i in holdout],\n                            gold=[rows[i]["gold"] for i in holdout],probs=probs)\n    return probs\n\n\ndef math_ceil(value):\n    import math\n    return math.ceil(value)\n\n\ndef choose_pilot(scores,control,reference_scores=()):\n    threshold=max([control,*reference_scores])\n    successful = [method for method in ("memory","evidence") if scores[method] > threshold]\n    if not successful:\n        return None\n    if len(successful)==2 and abs(scores["memory"]-scores["evidence"]) < .001:\n        return "evidence"\n    return max(successful,key=lambda method:scores[method])\n\n\ndef bootstrap_difference(gold,candidates,reference,count=1000,seed=42,calls=None):\n    gold,reference = np.asarray(gold),np.asarray(reference)\n    candidates = [np.asarray(values) for values in candidates]\n    def contributions(probability):\n        prediction = probability >= .5\n        return np.stack([prediction & (gold==1),prediction & (gold==0),(~prediction)&(gold==1)],axis=1).astype(np.int32)\n    ref = contributions(reference)\n    cand = [contributions(probability) for probability in candidates]\n    def score(counts):\n        tp,fp,fn = counts.sum(0)\n        return np.divide(2*tp,2*tp+fp+fn,out=np.zeros(9,dtype=float),where=(2*tp+fp+fn)>0).mean()\n    randomizer = np.random.default_rng(seed)\n    groups=None\n    if calls is not None:\n        if len(calls)!=len(gold):\n            raise ValueError("Bootstrap call identities must align with predictions")\n        if len(set(calls))!=len(calls):\n            buckets={}\n            for i,call in enumerate(calls):\n                buckets.setdefault(call,[]).append(i)\n            groups=list(buckets.values())\n    values = []\n    for _ in range(count):\n        indices = (randomizer.integers(0,len(gold),len(gold)) if groups is None else\n                   np.concatenate([groups[j] for j in randomizer.integers(0,len(groups),len(groups))]))\n        values.append(float(np.mean([score(item[indices]) for item in cand])-score(ref[indices])))\n    return [float(value) for value in np.quantile(values,[.025,.5,.975])]\n\n\ndef adoption_gate(gold,candidates,ensemble,controls,ablations,*,smoke=False,bootstrap=1000,calls=None):\n    base = metrics(gold,ensemble)\n    candidate = [metrics(gold,p) for p in candidates]\n    if len(candidate)!=3 or len(controls)!=3 or len(ablations)!=3:\n        raise ValueError("Adoption requires three candidates, controls and ablations")\n    mean = float(np.mean([m["macro_f1"] for m in candidate]))\n    interval = bootstrap_difference(gold,candidates,ensemble,bootstrap,calls=calls)\n    checks = dict(full_dataset=not smoke, mean_gain_ge_003=mean-base["macro_f1"] >= .003-1e-12,\n                  at_least_two_seeds_win=sum(m["macro_f1"]>base["macro_f1"] for m in candidate)>=2,\n                  ci_lower_positive=interval[0]>0,\n                  micro_drop_le_005=np.mean([m["micro_f1"] for m in candidate])>=base["micro_f1"]-.005)\n    contribution=dict(beats_additional_training_control=bool(mean>np.mean([metrics(gold,p)["macro_f1"] for p in controls])),\n                      beats_ablation=bool(mean>np.mean([metrics(gold,p)["macro_f1"] for p in ablations])))\n    return dict(passed=all(checks.values()),checks={k:bool(v) for k,v in checks.items()},\n                mean_macro_f1=mean, reference_macro_f1=base["macro_f1"],gain=mean-base["macro_f1"],\n                paired_bootstrap_ci95=interval,seed_metrics=candidate,contribution_checks=contribution)\n\n\ndef report(output,summary):\n    output=Path(output)\n    write_json(output/"metrics.json",summary)\n    lines=["# M3 context experiment", "",f"Status: **{summary[\'status\']}**", "",\n           "Threshold >= 0.5. Training-only retrieval. Official Validation excluded from fitting.","",\n           "| Model | Macro F1 | Micro F1 |","|---|---:|---:|"]\n    for name,item in summary.get("pilot",{}).items():\n        lines.append(f"| {name} | {item[\'macro_f1\']:.4f} | {item[\'micro_f1\']:.4f} |")\n    if "gate" in summary:\n        lines.extend(["", "## Adoption gate", "", "```json",json.dumps(summary["gate"],ensure_ascii=False,indent=2),"```"])\n    if summary.get("repeat"):\n        lines.extend(["", "## Three-seed comparison", "", "| Method | Stage | Mean Macro F1 | Mean Micro F1 |",\n                      "|---|---|---:|---:|"])\n        for method,stages in summary["repeat"].items():\n            for stage,items in stages.items():\n                lines.append(f"| {method} | {stage} | {np.mean([m[\'macro_f1\'] for m in items]):.4f} | "\n                             f"{np.mean([m[\'micro_f1\'] for m in items]):.4f} |")\n    if summary.get("smoke"):\n        lines.extend(["", "This bounded smoke run is an engineering check; its scores are not competition performance."])\n    (output/"report.md").write_text("\\n".join(lines)+"\\n",encoding="utf-8")\n    try:\n        import matplotlib\n        matplotlib.use("Agg")\n        import matplotlib.pyplot as plt\n        if summary.get("pilot"):\n            names=list(summary["pilot"])\n            fig,ax=plt.subplots(figsize=(8,4))\n            ax.barh(names,[summary["pilot"][name]["macro_f1"] for name in names])\n            ax.set(xlabel="Macro F1 at threshold 0.5",xlim=(0,1),\n                   title="M3 internal holdout comparison"+(" (smoke only)" if summary.get("smoke") else ""))\n            fig.tight_layout();fig.savefig(output/"comparison.png",dpi=180);plt.close(fig)\n    except ImportError:\n        pass\n\n\ndef run_experiment(args):\n    output=Path(args.out)\n    output.mkdir(parents=True,exist_ok=True)\n    budget=RunBudget(output,getattr(args,"max_hours",10))\n    try:\n        return _run_experiment(args,budget)\n    except BudgetReached:\n        path=output/"metrics.json"\n        summary=json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}\n        summary.update(status="budget_reached_resume_pending",selected_method=summary.get("selected_method"),\n                       smoke=bool(args.smoke_rows),elapsed_seconds=budget.used+time.monotonic()-budget.started)\n        report(output,summary)\n        return summary\n    finally:\n        try:\n            budget.check(force=True)\n        except BudgetReached:\n            pass\n\n\ndef _run_experiment(args,budget):\n    started=time.monotonic()\n    output=Path(args.out)\n    output.mkdir(parents=True,exist_ok=True)\n    device=torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))\n    torch.set_num_threads(args.cpu_threads)\n    rows,train,holdout,base_probs,ensemble=prepare_data(args.train_label_dir,args.reference_probs_dir,args.smoke_rows)\n    recipe=dict(RECIPE)\n    if args.smoke_rows:\n        recipe.update(head_epochs=2,bootstrap=100)\n    source=dict(base_checkpoint_sha256=file_hash(args.base_ckpt),\n                rows_sha256=digest([{k:row[k] for k in ("name","call","text_hash","gold")} for row in rows]),\n                split_sha256=digest(dict(train=train,holdout=holdout)),\n                recipe=recipe,context_config=context.DEFAULTS,smoke=bool(args.smoke_rows),\n                feature_precision="cuda_fp16" if device.type=="cuda" else "cpu_fp32",\n                implementation_sha256=digest({context.__name__:file_hash(context.__file__),\n                                              baseline.__name__:file_hash(baseline.__file__),\n                                              __name__:file_hash(__file__)}))\n    signature=digest(source)\n    previous=output/"run.json"\n    if previous.exists() and json.loads(previous.read_text(encoding="utf-8"))!=source:\n        raise ValueError("Output directory belongs to a different dataset/checkpoint/recipe")\n    write_json(previous,source)\n    seed_all(42)\n    model,tokenizer,payload=initialize(args.base_ckpt,"control",device)\n    if not args.smoke_rows and (payload.get("source",{}).get("exp_id")!="M3-ROBERTA-003" or payload["source"].get("seed")!=42):\n        raise ValueError("Full runs must start from ROBERTA-003 seed42")\n    indices=sorted(set(train+holdout))\n    features=extract_features(model,tokenizer,rows,indices,output/"features",signature,args.feature_batch,budget)\n    bank=make_bank(features,rows,train).to(device)\n    initial_base={key:value.detach().cpu().clone() for key,value in model.base.state_dict().items()}\n    gold=np.asarray([rows[i]["gold"] for i in holdout])\n    docs=[rows[i]["doc"] for i in holdout]\n    summary=dict(status="pilot",smoke=bool(args.smoke_rows),source=source,selected_method=None,\n                 device=str(device),pilot={"roberta003":metrics(gold,base_probs,docs),\n                                          "existing_ensemble":metrics(gold,ensemble,docs)})\n    recached=torch.stack([features[i]["base_logits"] for i in holdout]).sigmoid().numpy()\n    summary["pilot"]["recached_roberta003"]=metrics(gold,recached,docs)\n    summary["runtime"]=dict(torch=str(torch.__version__),numpy=np.__version__,\n                             gpu=torch.cuda.get_device_name(device) if device.type=="cuda" else None)\n    predictions={}\n    for method in ("control","memory","evidence"):\n        seed_all(42)\n        model.head=context.ResidualHead(model.base.classifier.in_features,method,context.DEFAULTS).to(device)\n        model.bank=bank if method.startswith("memory") else None\n        predictions[method]=fit_head(model,features,rows,train,holdout,42,output/f"{method}_42",signature,recipe,budget)\n        summary["pilot"][method]=metrics(gold,predictions[method],docs)\n        report(output,summary)\n    winner=choose_pilot({method:summary["pilot"][method]["macro_f1"] for method in ("memory","evidence")},\n                        summary["pilot"]["control"]["macro_f1"],\n                        [summary["pilot"][name]["macro_f1"] for name in ("roberta003","existing_ensemble")])\n    summary["selected_method"]=winner\n    if winner is None:\n        summary["status"]="both_pilots_failed"\n    elif args.phase=="pilot":\n        summary["status"]="pilot_complete_repeat_pending"\n    else:\n        ablation="memory_removed" if winner=="memory" else "answers_removed"\n        candidates,controls,ablations=[],[],[]\n        frozen_candidates,frozen_controls,frozen_ablations=[],[],[]\n        summary["repeat"]={method:dict(frozen=[],fine=[]) for method in ("control",winner,ablation)}\n        for seed in recipe["seeds"]:\n            for method,container in (("control",controls),(winner,candidates),(ablation,ablations)):\n                seed_all(seed)\n                model.base.load_state_dict(initial_base)\n                model.head=context.ResidualHead(model.base.classifier.in_features,method,context.DEFAULTS).to(device)\n                model.bank=bank if method.startswith("memory") else None\n                directory=output/f"{method}_{seed}"\n                frozen=fit_head(model,features,rows,train,holdout,seed,directory,signature,recipe,budget)\n                frozen_container={"control":frozen_controls,winner:frozen_candidates,ablation:frozen_ablations}[method]\n                frozen_container.append(frozen)\n                frozen_metrics=metrics(gold,frozen,docs)\n                summary["repeat"][method]["frozen"].append(frozen_metrics)\n                write_json(directory/"frozen_metrics.json",frozen_metrics)\n                probability=fit_encoder(model,tokenizer,rows,train,holdout,seed,directory,signature,recipe,budget)\n                container.append(probability)\n                fine_metrics=metrics(gold,probability,docs)\n                summary["repeat"][method]["fine"].append(fine_metrics)\n                write_json(directory/"fine_metrics.json",fine_metrics)\n                report(output,summary)\n        calls=[rows[i]["call"] for i in holdout]\n        summary["gate"]=adoption_gate(gold,candidates,ensemble,controls,ablations,smoke=bool(args.smoke_rows),bootstrap=recipe["bootstrap"],calls=calls)\n        summary["frozen_gate"]=adoption_gate(gold,frozen_candidates,ensemble,frozen_controls,frozen_ablations,\n                                             smoke=bool(args.smoke_rows),bootstrap=recipe["bootstrap"],calls=calls)\n        summary["status"]="internal_gate_passed" if summary["gate"]["passed"] else "internal_gate_failed"\n        report(output,summary)\n        if summary["gate"]["passed"] and args.phase=="all":\n            all_train=list(range(len(rows)))\n            # Holdout is now part of Training after recipe selection; official Val remains excluded.\n            seed_all(42)\n            model.base.load_state_dict(initial_base)\n            model.head=context.ResidualHead(model.base.classifier.in_features,winner,context.DEFAULTS).to(device)\n            model.bank=make_bank(features,rows,all_train).to(device) if winner=="memory" else None\n            final_dir=output/"final"\n            final_signature=digest(dict(source=source,training="all_29200",method=winner))\n            fit_head(model,features,rows,all_train,[],42,final_dir,final_signature,recipe,budget)\n            fit_encoder(model,tokenizer,rows,all_train,[],42,final_dir,final_signature,recipe,budget)\n            final_source=dict(source,representative_seed=42,training_rows=len(rows),selected_method=winner,\n                              internal_gate=summary["gate"],official_validation_used=False)\n            save_torch(final_dir/"candidate.pt",context.export_payload(model,payload,final_source))\n            write_json(final_dir/"candidate.pt.json",dict(source=final_source,\n                       checkpoint_sha256=file_hash(final_dir/"candidate.pt")))\n            summary["status"]="final_candidate_ready_official_pending"\n    summary["elapsed_seconds"]=time.monotonic()-started\n    summary["primary_metric"]=dict(name="macro_f1",value=max(summary["pilot"][name]["macro_f1"] for name in\n                                                          ("control","memory","evidence")),split="internal_holdout")\n    report(output,summary)\n    return summary\n', 'src/dcc_data/mission3_context_release.py': '"""One final official check and a separate, hash-verified offline package."""\nfrom __future__ import annotations\n\nimport json\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom dcc_inference import mission3_bert as baseline\nfrom dcc_inference import mission3_context as context\nfrom .mission3_context_experiment import digest, file_hash, metrics, write_json\n\n\ndef validate_final(payload):\n    source=payload.get("source",{})\n    gate=source.get("internal_gate",{})\n    old=payload.get("format")==context.FORMAT and payload.get("method") in ("memory","evidence")\n    new=payload.get("format")=="dcc-m3-evidence-v2" and payload.get("mode") in ("head_tail","multi","states")\n    loss=payload.get("format")==baseline.FORMAT and source.get("kind") in ("bce","asl")\n    if (not (old or new or loss)\n        or source.get("smoke",True) or source.get("training_rows")!=29200\n        or source.get("representative_seed")!=42 or source.get("official_validation_used") is not False\n        or not gate.get("passed") or not gate.get("checks") or not all(gate["checks"].values())):\n        raise ValueError("Official evaluation requires a full Training seed42 candidate with a passed internal gate")\n    if (new or loss) and (not gate.get("contribution_checks") or not all(gate["contribution_checks"].values())):\n        raise ValueError("New information/loss must beat its matched controls")\n\n\ndef candidate_probabilities(model,tokenizer,docs,names,batch_size):\n    if getattr(model,"checkpoint_format",None)=="dcc-m3-evidence-v2":\n        from dcc_inference.mission3_evidence import probabilities\n        return probabilities(model,tokenizer,docs,names,batch_size=batch_size)\n    if getattr(model,"requires_documents",False):\n        return context.probabilities(model,tokenizer,docs,names,batch_size=batch_size)\n    return baseline.probabilities(model,tokenizer,[baseline.transcript(doc) for doc in docs],batch_size=batch_size)\n\n\ndef evaluate_official(checkpoint,label_dir,output,registry,device=None,batch_size=32,expected_rows=3640):\n    """Reserve a single recipe; resume only this identical candidate and dataset.\n\n    Inference shards can resume after interruption. A completed evaluation returns\n    its existing result without running another forward pass. The registry is\n    shared across output directories, so a different directory does not reset it.\n    """\n    output,registry=Path(output),Path(registry)\n    payload=torch.load(checkpoint,map_location="cpu",weights_only=True)\n    validate_final(payload)\n    paths=sorted(Path(label_dir).rglob("*.json"))\n    if len(paths)!=expected_rows or len({p.name for p in paths})!=len(paths):\n        raise ValueError(f"Official Validation must contain {expected_rows} uniquely named calls")\n    docs=[json.loads(p.read_text(encoding="utf-8-sig")) for p in paths]\n    names=[p.name for p in paths]\n    gold=np.array([[int(label in doc.get("symptom",[])) for label in context.LABELS] for doc in docs])\n    specification=dict(checkpoint_sha256=file_hash(checkpoint),rows=len(paths),\n                       dataset_sha256=digest([dict(name=name,transcript=context.text_hash(context.text_and_spans(doc)[0]),\n                                                  gold=row.tolist()) for name,doc,row in zip(names,docs,gold)]))\n    registry.parent.mkdir(parents=True,exist_ok=True)\n    if registry.exists():\n        reserved=json.loads(registry.read_text(encoding="utf-8"))\n        if reserved["specification"]!=specification:\n            raise ValueError("Official Validation already reserved for another final candidate; no tuning against it")\n        output=Path(reserved["output"])\n        if reserved["status"]=="completed":\n            return json.loads((output/"official_metrics.json").read_text(encoding="utf-8"))\n    else:\n        with registry.open("x",encoding="utf-8") as stream:\n            json.dump(dict(specification=specification,status="in_progress",output=str(output.resolve())),stream,indent=2)\n    output.mkdir(parents=True,exist_ok=True)\n    model,tokenizer,_,_=baseline.load_checkpoint(checkpoint,device)\n    started=time.monotonic()\n    predictions=[]\n    for start in range(0,len(paths),batch_size):\n        shard=output/f"probabilities_{start:06d}.npz"\n        chunk_names=names[start:start+batch_size]\n        if shard.exists():\n            with np.load(shard,allow_pickle=False) as cached:\n                if cached["names"].astype(str).tolist()!=chunk_names:\n                    raise ValueError("Official inference shard order changed")\n                probs=cached["probs"].copy()\n        else:\n            probs=candidate_probabilities(model,tokenizer,docs[start:start+batch_size],chunk_names,batch_size)\n            np.savez_compressed(shard,names=chunk_names,probs=probs)\n        if probs.shape!=(len(chunk_names),9) or not np.isfinite(probs).all():\n            raise ValueError("Invalid official prediction shard")\n        predictions.append(probs)\n        if start%(batch_size*20)==0:\n            print(f"official final inference {min(start+batch_size,len(paths))}/{len(paths)}",flush=True)\n    probabilities=np.concatenate(predictions)\n    np.savez_compressed(output/"official.npz",names=names,gold=gold,probs=probabilities)\n    pd.DataFrame({"label file name":names,"symptom":[str([label for label,positive in zip(context.LABELS,row) if positive])\n                                                  for row in probabilities>=.5]})\\\n        .to_csv(output/"mission3.csv",index=False,encoding="utf-8")\n    result=dict(specification=specification,metrics=metrics(gold,probabilities,docs),\n                elapsed_seconds=time.monotonic()-started,threshold=.5,adoption_threshold=.6540)\n    result["adopt"]=bool(result["metrics"]["macro_f1"]>=.6540)\n    result["decision"]="replace_candidate" if result["adopt"] else "retain_roberta003"\n    write_json(output/"official_metrics.json",result)\n    write_json(registry,dict(specification=specification,status="completed",output=str(output.resolve()),adopt=result["adopt"]))\n    return result\n\n\ndef package_candidate(checkpoint,official_result,output,root):\n    checkpoint,output,root=Path(checkpoint),Path(output),Path(root)\n    payload=torch.load(checkpoint,map_location="cpu",weights_only=True)\n    validate_final(payload)\n    result=json.loads(Path(official_result).read_text(encoding="utf-8"))\n    if (not result.get("adopt") or result["metrics"]["macro_f1"]<.6540\n        or result["specification"]["checkpoint_sha256"]!=file_hash(checkpoint)):\n        raise ValueError("Package promotion requires a matching official score >= 0.6540")\n    if output.exists() and any(output.iterdir()):\n        raise ValueError("Use a new candidate directory; existing submission files are protected")\n    output.mkdir(parents=True,exist_ok=True)\n    legacy=payload.get("format")==context.FORMAT\n    template=root/"submission"/("mission3_context" if legacy else "mission3_evidence")\n    for name in ("inference.py","requirements.txt","README.md"):\n        shutil.copy2(template/name,output/name)\n    for name in (("mission3_bert.py","mission3_context.py") if legacy else ("mission3_bert.py","mission3_evidence.py")):\n        shutil.copy2(root/"src"/"dcc_inference"/name,output/name)\n    if not legacy:shutil.copy2(root/"src"/"dcc_data"/"mission3_evidence_data.py",output/"mission3_evidence_data.py")\n    shutil.copy2(checkpoint,output/"mission3.pt")\n    shutil.copy2(official_result,output/"official_metrics.json")\n    manifest={path.name:file_hash(path) for path in sorted(output.iterdir()) if path.is_file()}\n    write_json(output/"freeze.json",dict(method=payload.get("method",payload.get("mode",payload.get("source",{}).get("kind"))),seed=42,threshold=.5,\n                                       official_macro_f1=result["metrics"]["macro_f1"],files=manifest))\n    return output\n', 'src/dcc_inference/mission3_bert.py': '"""Mission 3 BERT-family classifier: self-contained checkpoint, export and probabilities.\n\nThe lab experiments (``scripts/lab/m3-kcbert-005.py`` and relatives) save\n``best.ckpt`` = ``{"config", "seed", "epoch", "model": state_dict, ...}`` where the\nstate dict belongs to ``Classifier`` below (``bert`` encoder + dropout + Linear on\nthe first token). Those files need the Hugging Face model folder next to them.\n:func:`export_checkpoint` folds the folder\'s config and tokenizer files into one\n``.pt`` so inference runs offline from that single file; :func:`load_checkpoint`\nand :func:`probabilities` use it.\n\nInput text is :func:`transcript`, the same rule the lab scripts train on:\n``utterances[].text`` only, whitespace collapsed, joined with one space.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any, Sequence\n\nimport numpy as np\n\nFORMAT = "dcc-m3-bert-v1"\nLABELS = ("고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란")\nMAX_LENGTH = 256\n# Weights are stored in the state dict; these files are not copied into the export.\n_WEIGHT_SUFFIXES = (".safetensors", ".bin", ".pt", ".pth", ".ckpt", ".h5", ".msgpack")\n_SKIP_FILES = {"dcc_revision.json"}\n\n\ndef transcript(doc: Any) -> str:\n    """Training text rule of the lab scripts; other JSON fields are never read."""\n\n    if not isinstance(doc, dict):\n        return ""\n    utterances = doc.get("utterances", [])\n    if not isinstance(utterances, list):\n        return ""\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict):\n            continue\n        text = utterance.get("text", "")\n        text = re.sub(r"\\s+", " ", text).strip() if isinstance(text, str) else ""\n        if text:\n            texts.append(text)\n    return " ".join(texts)\n\n\ndef read_text(json_path: str | Path) -> str:\n    with Path(json_path).open("r", encoding="utf-8-sig") as handle:\n        return transcript(json.load(handle))\n\n\ndef _classifier(config: Any, n_labels: int, dropout: float = 0.1):\n    import torch\n    from transformers import AutoModel\n\n    class Classifier(torch.nn.Module):\n        def __init__(self) -> None:\n            super().__init__()\n            self.bert = AutoModel.from_config(config)\n            self.dropout = torch.nn.Dropout(dropout)\n            self.classifier = torch.nn.Linear(config.hidden_size, n_labels)\n\n        def forward(self, ids, mask):\n            hidden = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0]\n            return self.classifier(self.dropout(hidden))\n\n    return Classifier()\n\n\ndef _text_files(model_dir: Path) -> dict[str, str]:\n    files: dict[str, str] = {}\n    for path in sorted(model_dir.iterdir()):\n        if not path.is_file() or path.name in _SKIP_FILES or path.suffix in _WEIGHT_SUFFIXES:\n            continue\n        try:\n            files[path.name] = path.read_text(encoding="utf-8")\n        except UnicodeDecodeError as error:\n            raise ValueError(f"tokenizer/config file is not text, cannot embed: {path}") from error\n    if "config.json" not in files:\n        raise FileNotFoundError(f"config.json not found in {model_dir}")\n    return files\n\n\ndef export_checkpoint(lab_ckpt: str | Path, model_dir: str | Path, output: str | Path) -> dict:\n    """Write a self-contained inference checkpoint; returns its metadata (no tensors)."""\n\n    import torch\n\n    lab_ckpt, model_dir, output = Path(lab_ckpt), Path(model_dir), Path(output)\n    source = torch.load(lab_ckpt, map_location="cpu", weights_only=False)\n    config = source.get("config", {})\n    labels = tuple(config.get("labels", ()))\n    if labels != LABELS:\n        raise ValueError(f"checkpoint labels {labels} != Mission 3 labels {LABELS}")\n    if int(config.get("max_length", -1)) != MAX_LENGTH:\n        raise ValueError(f"checkpoint max_length {config.get(\'max_length\')} != {MAX_LENGTH}")\n    files = _text_files(model_dir)\n    digest = hashlib.sha256()\n    with lab_ckpt.open("rb") as handle:\n        for block in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(block)\n    meta = dict(\n        format=FORMAT,\n        labels=list(LABELS),\n        max_length=MAX_LENGTH,\n        dropout=float(config.get("dropout", 0.1)),\n        source=dict(exp_id=config.get("exp_id"), model=config.get("model"),\n                    revision=config.get("revision"), seed=source.get("seed"),\n                    epoch=source.get("epoch"), lab_ckpt_sha256=digest.hexdigest()),\n    )\n    state = {key: value.contiguous() for key, value in source["model"].items()}\n    # Fail now, not at submission time, if the state dict does not fit the classifier.\n    _build(files, meta).load_state_dict(state, strict=True)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(dict(meta, files=files, model=state), output)\n    return meta\n\n\ndef _build(files: dict[str, str], meta: dict):\n    from transformers import AutoConfig\n\n    with tempfile.TemporaryDirectory() as folder:\n        (Path(folder) / "config.json").write_text(files["config.json"], encoding="utf-8")\n        config = AutoConfig.from_pretrained(folder, local_files_only=True)\n    return _classifier(config, len(meta["labels"]), meta.get("dropout", 0.1))\n\n\ndef is_bert_checkpoint(path: str | Path) -> bool:\n    return Path(path).suffix.lower() in {".pt", ".pth", ".ckpt"}\n\n\ndef load_checkpoint(path: str | Path, device: str | None = None):\n    """``(model, tokenizer, labels, max_length)`` from an exported checkpoint, offline."""\n\n    import torch\n    from transformers import AutoTokenizer\n\n    payload = torch.load(Path(path), map_location="cpu", weights_only=True)\n    if isinstance(payload, dict) and payload.get("format") == "dcc-m3-evidence-v2":\n        try:\n            from .mission3_evidence import load_payload\n        except ImportError:\n            from mission3_evidence import load_payload\n        return load_payload(payload, device)\n    if isinstance(payload, dict) and payload.get("format") == "dcc-m3-context-v1":\n        try:\n            from .mission3_context import load_payload\n        except ImportError:  # offline standalone package\n            from mission3_context import load_payload\n        return load_payload(payload, device)\n    if not isinstance(payload, dict) or payload.get("format") != FORMAT:\n        raise ValueError(\n            f"{path} is not a {FORMAT} checkpoint; export the lab best.ckpt with "\n            "scripts/mission3_export_bert_ckpt.py"\n        )\n    labels = [str(label) for label in payload["labels"]]\n    if tuple(labels) != LABELS:\n        raise ValueError(f"checkpoint labels {labels} != Mission 3 labels {list(LABELS)}")\n    files = payload["files"]\n    with tempfile.TemporaryDirectory() as folder:\n        for name, text in files.items():\n            (Path(folder) / Path(name).name).write_text(text, encoding="utf-8")\n        tokenizer = AutoTokenizer.from_pretrained(folder, local_files_only=True)\n    model = _build(files, payload)\n    model.load_state_dict(payload["model"], strict=True)\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    model.to(device).eval()\n    return model, tokenizer, labels, int(payload["max_length"])\n\n\ndef probabilities(model, tokenizer, texts: Sequence[str], max_length: int = MAX_LENGTH,\n                  batch_size: int = 32) -> np.ndarray:\n    """Sigmoid probabilities ``(len(texts), n_labels)`` in input order (fp32)."""\n\n    import torch\n\n    if getattr(model, "requires_documents", False):\n        raise ValueError("Use mission3_context.probabilities with raw documents and file names; "\n                         "joined strings lose utterance boundaries and retrieval exclusions")\n    device = next(model.parameters()).device\n    # Sort by length so each padded batch is short; results go back to input order.\n    order = sorted(range(len(texts)), key=lambda index: len(texts[index]))\n    out = np.zeros((len(texts), model.classifier.out_features), dtype=np.float32)\n    with torch.inference_mode():\n        for start in range(0, len(order), batch_size):\n            chunk = order[start:start + batch_size]\n            encoded = tokenizer([texts[index] for index in chunk], padding=True, truncation=True,\n                                max_length=max_length, return_tensors="pt")\n            logits = model(encoded["input_ids"].to(device), encoded["attention_mask"].to(device))\n            out[chunk] = logits.float().sigmoid().cpu().numpy()\n    return out\n', 'src/dcc_inference/mission3_context.py': '"""Text-only M3 residual models: contrastive memory and adjacent-turn evidence.\n\nThe memory contains Training representations, never validation/test annotations.\nAll models retain sigmoid >= .5, including empty predictions. This module is also\ncopied beside mission3_bert.py when creating an offline submission package.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nimport tempfile\nimport unicodedata\nfrom pathlib import Path\nfrom typing import Any, Sequence\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\ntry:\n    from . import mission3_bert as baseline\nexcept ImportError:  # standalone candidate package\n    import mission3_bert as baseline\n\nFORMAT = "dcc-m3-context-v1"\nLABELS = baseline.LABELS\nMETHODS = ("control", "memory", "evidence", "memory_removed", "answers_removed")\nDEFAULTS = dict(dimension=128, dropout=.1, positive_k=8, negative_k=8,\n                overall_k=32, temperature=.1, max_length=256)\n\n\ndef text_and_spans(doc: Any) -> tuple[str, list[tuple[int, int]]]:\n    """Exactly the baseline text; recover boundaries without adding any tokens."""\n    utterances = doc.get("utterances", []) if isinstance(doc, dict) else []\n    if not isinstance(utterances, list):\n        utterances = []\n    parts, spans, length = [], [], 0\n    for utterance in utterances:\n        text = utterance.get("text", "") if isinstance(utterance, dict) else ""\n        text = re.sub(r"\\s+", " ", text).strip() if isinstance(text, str) else ""\n        if text:\n            start = length + bool(parts)\n            spans.append((start, start + len(text)))\n            parts.append(text)\n            length = start + len(text)\n    return " ".join(parts), spans\n\n\ndef text_hash(text: str) -> str:\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n\n\ndef call_key(name: str) -> str:\n    """Paired file stem, without the date suffix; identity is never a feature."""\n    stem = Path(unicodedata.normalize("NFC", name)).stem\n    return re.sub(r"_\\d{8}$", "", stem)\n\n\ndef tokenize_documents(tokenizer, documents: Sequence[dict], max_length: int = 256):\n    if not tokenizer.is_fast:\n        raise ValueError("M3 context requires a fast tokenizer with character offsets")\n    texts_spans = [text_and_spans(doc) for doc in documents]\n    encoded = tokenizer([text for text, _ in texts_spans], padding=True, truncation=True,\n                        max_length=max_length, return_offsets_mapping=True,\n                        return_tensors="pt")\n    offsets = encoded.pop("offset_mapping")\n    turn_ids = torch.full(encoded["input_ids"].shape, -1, dtype=torch.long)\n    for row, (_, spans) in enumerate(texts_spans):\n        cursor = 0\n        for token, (start, end) in enumerate(offsets[row].tolist()):\n            if end <= start:\n                continue  # CLS/SEP/padding, including the empty transcript\n            while cursor < len(spans) and spans[cursor][1] <= start:\n                cursor += 1\n            if cursor < len(spans) and start < spans[cursor][1] and end > spans[cursor][0]:\n                turn_ids[row, token] = cursor\n    encoded["turn_ids"] = turn_ids\n    return encoded, [text_hash(text) for text, _ in texts_spans]\n\n\ndef pool_turns(hidden: torch.Tensor, turn_ids: torch.Tensor):\n    """Mean contextual token representations for each visible utterance."""\n    count = max(1, int(turn_ids.max().item()) + 1)\n    index = turn_ids.clamp_min(0)\n    visible = turn_ids >= 0\n    values = hidden.float() * visible.unsqueeze(-1)\n    sums = torch.zeros(hidden.shape[0], count, hidden.shape[-1], device=hidden.device)\n    sums.scatter_add_(1, index.unsqueeze(-1).expand_as(hidden), values)\n    counts = torch.zeros(hidden.shape[0], count, device=hidden.device)\n    counts.scatter_add_(1, index, visible.float())\n    return sums / counts.clamp_min(1).unsqueeze(-1), counts > 0\n\n\nclass MemoryBank(nn.Module):\n    def __init__(self, vectors, labels, calls, hashes, *, split="training"):\n        super().__init__()\n        if split != "training":\n            raise ValueError("Only Training may populate the retrieval bank")\n        vectors = torch.as_tensor(vectors).float()\n        labels = torch.as_tensor(labels).float()\n        if vectors.ndim != 2 or labels.shape != (len(vectors), 9):\n            raise ValueError("Invalid memory vector/label shapes")\n        if len(calls) != len(vectors) or len(hashes) != len(vectors):\n            raise ValueError("Memory identities must align with vectors")\n        if not torch.isfinite(vectors).all() or not torch.isfinite(labels).all():\n            raise ValueError("Non-finite memory")\n        if not ((labels == 0) | (labels == 1)).all():\n            raise ValueError("Memory labels must be binary")\n        self.register_buffer("vectors", vectors)\n        self.register_buffer("keys", F.normalize(vectors, dim=-1))\n        self.register_buffer("labels", labels)\n        self.calls, self.hashes = list(calls), list(hashes)\n        self.exclusions: dict[str, list[int]] = {}\n        for i, (call, digest) in enumerate(zip(calls, hashes)):\n            self.exclusions.setdefault("c:" + call, []).append(i)\n            self.exclusions.setdefault("h:" + digest, []).append(i)\n\n    def similarities(self, queries, calls, hashes):\n        if len(calls) != len(queries) or len(hashes) != len(queries):\n            raise ValueError("Every query needs its call and transcript identity")\n        scores = F.normalize(queries.float(), dim=-1) @ self.keys.T\n        for i, (call, digest) in enumerate(zip(calls, hashes)):\n            excluded = self.exclusions.get("c:" + call, []) + self.exclusions.get("h:" + digest, [])\n            if excluded:\n                scores[i, excluded] = -torch.inf\n        return scores\n\n    def _neighbors(self, scores, k, temperature):\n        if scores.shape[1] == 0:\n            shape = (len(scores), 0)\n            return scores.new_zeros(shape), torch.empty(shape, dtype=torch.long, device=scores.device)\n        values, indices = scores.topk(min(k, scores.shape[1]), dim=-1)\n        valid = torch.isfinite(values)\n        # All-excluded neighborhoods must have zero weight, not NaN softmax.\n        weights = (values.masked_fill(~valid, -1e4) / temperature).softmax(-1) * valid\n        weights = weights / weights.sum(-1, keepdim=True).clamp_min(1e-8)\n        return weights, indices\n\n    def retrieve(self, queries, calls, hashes, config):\n        scores = self.similarities(queries, calls, hashes)\n        weight, index = self._neighbors(scores, config["overall_k"], config["temperature"])\n        vote = (self.labels[index] * weight.unsqueeze(-1)).sum(1)\n        entropy = -(vote.clamp(1e-6, 1-1e-6) * vote.clamp(1e-6, 1-1e-6).log()\n                    + (1-vote).clamp(1e-6, 1-1e-6) * (1-vote).clamp(1e-6, 1-1e-6).log()) / math.log(2)\n        prototypes, statistics = [], []\n        for label in range(9):\n            pair, means, availability = [], [], []\n            for positive, k in ((True, config["positive_k"]), (False, config["negative_k"])):\n                eligible = self.labels[:, label].bool() == positive\n                subset = scores.masked_fill(~eligible.unsqueeze(0), -torch.inf)\n                w, idx = self._neighbors(subset, k, config["temperature"])\n                pair.append((self.vectors[idx] * w.unsqueeze(-1)).sum(1))\n                means.append((subset.gather(1, idx).masked_fill(w == 0, 0) * w).sum(1))\n                availability.append((w.sum(1) > 0).float())\n            prototypes.append(torch.stack(pair, 1))\n            statistics.append(torch.stack([means[0], means[1], means[0]-means[1],\n                                           vote[:, label], entropy[:, label],\n                                           availability[0]*availability[1]], -1))\n        return torch.stack(prototypes, 1), torch.stack(statistics, 1)\n\n\nclass ResidualHead(nn.Module):\n    def __init__(self, hidden_size, method, config):\n        super().__init__()\n        if method not in METHODS:\n            raise ValueError(f"Unknown M3 method: {method}")\n        self.method, self.config = method, dict(config)\n        dimension = config["dimension"]\n        self.project = nn.Linear(hidden_size, dimension)\n        self.dropout = nn.Dropout(config["dropout"])\n        self.label_queries = nn.Parameter(torch.randn(9, dimension) / math.sqrt(dimension))\n        self.window = nn.Linear(3 * hidden_size, dimension)\n        width = 4 * dimension + 6 if method.startswith("memory") else 2 * dimension\n        self.residual = nn.Sequential(nn.Linear(width, dimension), nn.ReLU(),\n                                      nn.Dropout(config["dropout"]), nn.Linear(dimension, 1))\n        self.gate = nn.Linear(6 if method.startswith("memory") else dimension, 1)\n        nn.init.zeros_(self.residual[-1].weight)\n        nn.init.zeros_(self.residual[-1].bias)\n\n    def forward(self, cls, base_logits, turns, turn_mask, *, bank=None, calls=(), hashes=()):\n        query = self.dropout(torch.tanh(self.project(cls.float())))\n        query = query[:, None, :].expand(-1, 9, -1)\n        if self.method.startswith("memory"):\n            if bank is None:\n                raise ValueError("Memory method requires a Training bank")\n            if self.method == "memory_removed":\n                prototypes = cls.new_zeros(len(cls),9,2,cls.shape[-1])\n                stats = cls.new_zeros(len(cls),9,6)\n            else:\n                prototypes, stats = bank.retrieve(cls, calls, hashes, self.config)\n            projected = torch.tanh(self.project(prototypes))\n            labels = self.label_queries[None].expand(len(cls), -1, -1)\n            inputs = torch.cat([query, projected[:, :, 0], projected[:, :, 1], labels, stats], -1)\n            gate = self.gate(stats).sigmoid()\n            supported = (stats[:, :, 5] > 0).unsqueeze(-1) if self.method == "memory" else 1\n        elif self.method in ("evidence", "answers_removed"):\n            padded = F.pad(turns.float()*turn_mask.unsqueeze(-1), (0, 0, 0, 2))\n            following = [padded[:, i:i+turns.shape[1]] for i in range(3)]\n            if self.method == "answers_removed":\n                following[1:] = [torch.zeros_like(following[0]), torch.zeros_like(following[0])]\n            windows = torch.tanh(self.window(torch.cat(following, -1)))\n            attention = torch.einsum("btd,ld->blt", windows, self.label_queries) / math.sqrt(query.shape[-1])\n            attention = attention.masked_fill(~turn_mask[:, None, :], -1e4).softmax(-1)\n            attention = attention * turn_mask[:, None, :]\n            attention = attention / attention.sum(-1, keepdim=True).clamp_min(1e-8)\n            evidence = torch.einsum("blt,btd->bld", attention, windows)\n            inputs = torch.cat([query, evidence], -1)\n            gate = self.gate(evidence).sigmoid()\n            supported = turn_mask.any(-1)[:, None, None]\n        else:\n            inputs = torch.cat([query, self.label_queries[None].expand(len(cls), -1, -1)], -1)\n            gate, supported = self.gate(query).sigmoid(), 1\n        return base_logits.float() + (self.residual(inputs) * gate * supported).squeeze(-1)\n\n\nclass ContextModel(nn.Module):\n    requires_documents = True\n\n    def __init__(self, base, method, config=None, bank=None):\n        super().__init__()\n        self.base = base\n        self.config = dict(DEFAULTS, **(config or {}))\n        self.head = ResidualHead(base.classifier.in_features, method, self.config)\n        self.bank = bank\n\n    def extract(self, ids, mask, turn_ids):\n        hidden = self.base.bert(input_ids=ids, attention_mask=mask).last_hidden_state\n        cls = hidden[:, 0].float()\n        turns, turn_mask = pool_turns(hidden, turn_ids)\n        logits = self.base.classifier(self.base.dropout(hidden[:, 0])).float()\n        return cls, logits, turns, turn_mask\n\n    def forward(self, ids, mask, turn_ids, calls=(), hashes=()):\n        cls, logits, turns, turn_mask = self.extract(ids, mask, turn_ids)\n        # Keep the residual module in fp32 under the encoder\'s AMP context.\n        with torch.autocast(device_type=ids.device.type, enabled=False):\n            return self.head(cls, logits, turns, turn_mask, bank=self.bank, calls=calls, hashes=hashes)\n\n\ndef load_payload(payload, device=None):\n    from transformers import AutoTokenizer\n    if payload.get("format") != FORMAT or tuple(payload.get("labels", ())) != LABELS:\n        raise ValueError("Invalid M3 context checkpoint format/labels")\n    config = dict(payload["context_config"])\n    if config["max_length"] != 256 or payload.get("threshold", .5) != .5:\n        raise ValueError("M3 context requires max_length=256 and threshold=0.5")\n    files = payload["files"]\n    with tempfile.TemporaryDirectory() as folder:\n        for name, text in files.items():\n            (Path(folder) / Path(name).name).write_text(text, encoding="utf-8")\n        tokenizer = AutoTokenizer.from_pretrained(folder, local_files_only=True, use_fast=True)\n    base = baseline._build(files, payload)\n    memory = payload.get("memory")\n    if payload["method"].startswith("memory") and memory is None:\n        raise ValueError("A memory checkpoint must include its Training bank")\n    bank = MemoryBank(**memory) if memory is not None else None\n    model = ContextModel(base, payload["method"], config, bank)\n    # The bank is serialized separately, so its derived keys are not duplicated.\n    state = model.state_dict()\n    state.update(payload["model"])\n    expected = {key for key in state if not key.startswith("bank.")}\n    if set(payload["model"]) != expected:\n        raise ValueError("Context checkpoint model keys do not match architecture")\n    model.load_state_dict(state, strict=True)\n    model.to(device or ("cuda" if torch.cuda.is_available() else "cpu")).eval()\n    return model, tokenizer, list(LABELS), 256\n\n\ndef export_payload(model, base_payload, source):\n    memory = None\n    if model.bank is not None:\n        memory = dict(vectors=model.bank.vectors.detach().cpu(), labels=model.bank.labels.detach().cpu(),\n                      calls=model.bank.calls, hashes=model.bank.hashes, split="training")\n    return dict(format=FORMAT, files=base_payload["files"], labels=list(LABELS), max_length=256,\n                dropout=base_payload.get("dropout", .1), threshold=.5,\n                method=model.head.method, context_config=model.config, source=source,\n                memory=memory, model={k:v.detach().cpu().contiguous() for k,v in model.state_dict().items()\n                                     if not k.startswith("bank.")})\n\n\ndef probabilities(model, tokenizer, documents, names, max_length=256, batch_size=32):\n    if len(names) != len(documents):\n        raise ValueError("File names and documents must align")\n    device = next(model.parameters()).device\n    out = np.zeros((len(documents), 9), dtype=np.float32)\n    with torch.inference_mode():\n        for start in range(0, len(documents), batch_size):\n            docs, chunk_names = documents[start:start+batch_size], names[start:start+batch_size]\n            encoded, hashes = tokenize_documents(tokenizer, docs, max_length)\n            tensors = {k:v.to(device) for k,v in encoded.items()}\n            logits = model(tensors["input_ids"], tensors["attention_mask"], tensors["turn_ids"],\n                           [call_key(name) for name in chunk_names], hashes)\n            out[start:start+len(docs)] = logits.sigmoid().cpu().numpy()\n    return out\n', 'src/dcc_data/mission3_evidence_data.py': '"""Text-only evidence windows and conservative Training weak supervision.\n\nWeak states describe a local window, never the absence of a call-level symptom.\nThey do not use call labels, speaker IDs, or any patient metadata.\n"""\nfrom __future__ import annotations\nimport re\ntry:\n    from dcc_inference.mission3_bert import LABELS, transcript\nexcept ImportError:\n    from mission3_bert import LABELS, transcript\n\nIGNORE = -100\nPOSITIVE, NEGATIVE, UNKNOWN = 0, 1, 2\nQUESTION = re.compile(r"\\?|(?<!주)세요|나요|습니까|어때|어떻|있으신|있는지")\nYES = re.compile(r"^(네|예|응|맞아|맞습니다|있어요|있습니다)[\\s,.!~]*$")\nNO = re.compile(r"^(아니요|아니오|아뇨|아니|없어요|없습니다)[\\s,.!~]*$")\n# Auxiliary states use narrower cues than the legacy diagnostic regex.\n# "호흡", "배가", and "피가" alone do not assert a symptom.\nSTATE_KEYWORDS = {\n    "고열":r"고열|열이?\\s*(?:많|높|나|있)|체온.{0,5}(?:38|39|40|41)",\n    "구토":r"토했|토해|토하|토를|구토|게워|오바이트",\n    "두통":r"머리.{0,8}(?:아프|아파|아퍼|깨질|지끈|띵|쑤)|두통|편두통",\n    "복통":r"배.{0,6}(?:아프|아파|아퍼)|복통|명치.{0,5}(?:아프|아파)",\n    "어지러움":r"어지러|어지럽|어질|빙빙|핑\\s*돌|현기증",\n    "열상":r"찢어|베였|베이|피가?\\s*(?:나|흐르)|출혈|꿰매|상처",\n    "오심":r"메스껍|메슥|울렁|구역|토할\\s*(?:것|거)|속이\\s*(?:안\\s*좋|이상|불편|뒤집)|오심",\n    "전신쇠약":r"기운이?\\s*없|힘이?\\s*없|기력이?\\s*없|쳐지|처지|축\\s*늘어|무기력|못\\s*일어|쇠약",\n    "호흡곤란":r"호흡곤란|숨.{0,6}(?:못|안\\s*쉬|차|막|가쁘|가빠|헐떡|힘들)|호흡.{0,5}(?:어렵|곤란|힘들)"}\n\n\ndef utterance_spans(doc):\n    parts = []\n    utterances = doc.get("utterances", []) if isinstance(doc, dict) else []\n    for item in utterances if isinstance(utterances,list) else []:\n        text = item.get("text", "") if isinstance(item, dict) else ""\n        text = re.sub(r"\\s+", " ", text).strip() if isinstance(text, str) else ""\n        if text:\n            parts.append(text)\n    spans, cursor = [], 0\n    for text in parts:\n        spans.append((cursor, cursor + len(text)))\n        cursor += len(text) + 1\n    return parts, spans\n\n\ndef weak_states(parts, keywords=None):\n    """Conservative weak rules; conflicting evidence is ignored."""\n    keywords=keywords or STATE_KEYWORDS\n    states = []\n    for label in LABELS:\n        cue = re.compile(keywords[label])\n        found = set()\n        for i, text in enumerate(parts):\n            matches = list(cue.finditer(text))\n            if not matches:\n                continue\n            if QUESTION.search(text):\n                answers = []\n                for following in parts[i+1:i+3]:\n                    if QUESTION.search(following):break\n                    answers.append(following)\n                yes = any(YES.fullmatch(answer) for answer in answers)\n                no = any(NO.fullmatch(answer) for answer in answers)\n                if yes and not no:\n                    found.add(POSITIVE)\n                elif no and not yes:\n                    found.add(NEGATIVE)\n                elif not yes and not no:\n                    found.add(UNKNOWN)\n                else:\n                    found.update((POSITIVE, NEGATIVE))\n            else:\n                for match in matches:\n                    # Absence inside "기력이 없" or "속이 안 좋" asserts a symptom.\n                    # Ignore unrelated preceding "없고" and discourse "아니".\n                    matched=match.group()\n                    tail=text[match.end():match.end()+14]\n                    embedded=re.search(r"안\\s*(?:아프|아파|울렁|메스)|지\\s*않",matched)\n                    following=re.match(r"\\s*(?:(?:은|는|이|가|도)\\s*)?(?:없|아니|지\\s*않|하지\\s*않)",tail)\n                    found.add(NEGATIVE if embedded or following else POSITIVE)\n        decisive = found - {UNKNOWN}\n        states.append(next(iter(decisive)) if len(decisive) == 1 else\n                      UNKNOWN if found == {UNKNOWN} else IGNORE)\n    return states\n\n\ndef tokenize_views(tokenizer, doc, mode="multi", max_length=256, max_windows=6):\n    """Return token windows and original character ranges; first view stays exact."""\n    if mode not in ("front", "head_tail", "multi", "states", "states_removed"):\n        raise ValueError(f"Unknown input mode: {mode}")\n    text = transcript(doc)\n    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)\n    ids, offsets = encoded["input_ids"], encoded["offset_mapping"]\n    capacity = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n    if capacity < 4 or max_windows < 2:\n        raise ValueError("Window capacity >=4 and max_windows >=2 required")\n    def view(start, end):\n        char_range = (offsets[start][0], offsets[end-1][1]) if end > start else (0, 0)\n        return dict(ids=tokenizer.build_inputs_with_special_tokens(ids[start:end]),\n                    token_range=(start, end), char_range=char_range)\n    front = view(0, min(capacity, len(ids)))\n    if mode == "front" or len(ids) <= capacity:\n        return [front]\n    if mode == "head_tail":\n        if tokenizer.sep_token_id is None:\n            raise ValueError("head_tail requires an explicit separator")\n        left = (capacity-1+1)//2\n        right = capacity-1-left\n        return [dict(ids=tokenizer.build_inputs_with_special_tokens(\n                         ids[:left] + [tokenizer.sep_token_id] + ids[-right:]),\n                     token_ranges=((0, left), (len(ids)-right, len(ids))),\n                     char_range=None)]\n    _, spans = utterance_spans(doc)\n    ends = {end for _, end in spans}\n    starts = {start for start, _ in spans}\n    # Offset boundaries remain those of the exact normalized full transcript.\n    boundaries = [i+1 for i, (_, end) in enumerate(offsets) if end in ends]\n    turn_starts = [i for i, (start, _) in enumerate(offsets) if start in starts]\n    chunks, start = [front], 0\n    while start < len(ids):\n        limit = min(start+capacity, len(ids))\n        candidates = [end for end in boundaries if start < end <= limit]\n        end = max(candidates) if candidates and limit < len(ids) else limit\n        chunks.append(view(start, end))\n        if end == len(ids):\n            break\n        preceding = [i for i in turn_starts if start < i < end]\n        overlap = preceding[-2] if len(preceding) >= 2 else end\n        start = overlap if overlap > start else end\n    unique, seen = [], set()\n    for chunk in chunks:\n        if chunk["token_range"] not in seen:\n            unique.append(chunk); seen.add(chunk["token_range"])\n    if len(unique) > max_windows:\n        # Bounded uniform selection retains front and final evidence; report coverage.\n        positions = sorted({round(i*(len(unique)-1)/(max_windows-1)) for i in range(max_windows)})\n        unique = [unique[i] for i in positions]\n    return unique\n\n\ndef window_states(doc, views, keywords):\n    parts, spans = utterance_spans(doc)\n    result = []\n    for view in views:\n        region = view.get("char_range")\n        if region is None:\n            result.append([IGNORE]*len(LABELS)); continue\n        a, b = region\n        # Supervision only from fully visible utterances; truncated answers cannot label a window.\n        visible = [part for part, (start, end) in zip(parts, spans) if a <= start and end <= b]\n        result.append(weak_states(visible, keywords))\n    return result\n', 'src/dcc_data/mission3_next_experiment.py': '"""Controlled M3 loss, coverage, and local evidence-state experiments.\n\nOnly Training is loaded. Full frozen and fine pilots are measured at fixed\nepochs; official Validation is reserved for a separately gated final candidate.\n"""\nfrom __future__ import annotations\nimport copy\nimport csv\nimport json\nimport re\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom . import mission3_context_experiment as exp\nfrom . import mission3_evidence_data as data\nfrom dcc_inference import mission3_bert as baseline, mission3_evidence as evidence\n\nRECIPE = dict(head_epochs=10, head_batch=128, head_lr=.001, encoder_epochs=1,\n              encoder_lr=1e-6, head_fine_lr=1e-4, effective_batch=16,\n              base_micro_batch=8, window_micro_batch=2, state_weight=.1,\n              weight_decay=.01, gamma_pos=0., gamma_neg=4., asl_clip=.05,\n              threshold=.5, seeds=[42,43,44], max_windows=6)\nINPUT_METHODS = ("front", "head_tail", "multi", "states", "states_removed")\n\n\ndef objective(logits, targets, weights, kind):\n    logits, targets = logits.float(), targets.float()\n    if kind == "original":\n        return exp.loss(logits, targets, weights)\n    if kind == "bce":\n        return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=weights)\n    if kind != "asl":\n        raise ValueError(f"Unknown loss: {kind}")\n    positive = logits.sigmoid()\n    negative = (1-positive+RECIPE["asl_clip"]).clamp(max=1)\n    weight = (1-positive).pow(RECIPE["gamma_pos"])*targets + (1-negative).pow(RECIPE["gamma_neg"])*(1-targets)\n    log_likelihood = targets*positive.clamp_min(1e-8).log()+(1-targets)*negative.clamp_min(1e-8).log()\n    return -(weight.detach()*log_likelihood).mean()\n\n\ndef auxiliary_loss(state_logits, states):\n    valid = states != data.IGNORE\n    if not valid.any():\n        return state_logits.sum()*0\n    return F.cross_entropy(state_logits[valid].float(), states[valid])\n\n\ndef source_signature(rows, train, holdout, checkpoint, smoke, device="cpu"):\n    source = dict(base_checkpoint_sha256=exp.file_hash(checkpoint),\n                  rows_sha256=exp.digest([{k: row[k] for k in ("name","call","text_hash","gold")} for row in rows]),\n                  split_sha256=exp.digest(dict(train=train, holdout=holdout)),\n                  recipe=RECIPE, config=evidence.DEFAULTS, smoke=bool(smoke),\n                  feature_precision="cuda_fp16" if str(device).startswith("cuda") else "cpu_fp32",\n                  implementation_sha256=exp.digest({module.__name__:exp.file_hash(module.__file__)\n                      for module in (exp, data, evidence, baseline, __import__(__name__, fromlist=[\'\']))}))\n    return source, exp.digest(source)\n\n\ndef inspect_coverage(rows, train, holdout, tokenizer, base_probs, out, budget):\n    """Token-location diagnostics are lexical proxies, not annotated symptom evidence."""\n    out = Path(out)\n    if (out/"coverage.json").exists():\n        return json.loads((out/"coverage.json").read_text(encoding="utf-8"))\n    holdout_position = {index: j for j,index in enumerate(holdout)}\n    counters = {label:dict(false_negative=0, no_lexical_cue=0, cue_only_after_front=0,\n                           cue_in_front=0) for label in baseline.LABELS}\n    lengths, truncated, weak_counts, audit = [], 0, np.zeros((9,3),dtype=int), []\n    sampled=np.zeros((9,3),dtype=int)\n    training = set(train)\n    for index, row in enumerate(rows):\n        if index % 1000 == 0:\n            budget.check(); print(f"coverage audit {index}/{len(rows)}", flush=True)\n        text = baseline.transcript(row["doc"])\n        encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)\n        lengths.append(len(encoded["input_ids"]))\n        capacity = 256-tokenizer.num_special_tokens_to_add(pair=False)\n        offsets = encoded["offset_mapping"]\n        cutoff = offsets[min(capacity,len(offsets))-1][1] if offsets else 0\n        truncated += len(offsets)>capacity\n        if index in holdout_position:\n            j = holdout_position[index]\n            for label_index,label in enumerate(baseline.LABELS):\n                if row["gold"][label_index] and base_probs[j,label_index]<.5:\n                    item = counters[label]; item["false_negative"] += 1\n                    matches = list(re.finditer(exp.KEYWORDS[label], text))\n                    if not matches:\n                        item["no_lexical_cue"] += 1\n                    elif all(match.start()>=cutoff for match in matches):\n                        item["cue_only_after_front"] += 1\n                    else:\n                        item["cue_in_front"] += 1\n        # Aux annotation candidates come only from the existing Training split.\n        if index in training:\n            parts, _ = data.utterance_spans(row["doc"])\n            for start in range(len(parts)):\n                window = parts[start:start+3]\n                states = data.weak_states(window)\n                for label_index,state in enumerate(states):\n                    if state != data.IGNORE:\n                        weak_counts[label_index,state] += 1\n                        if sampled[label_index,state]<3:\n                            sampled[label_index,state]+=1\n                            audit.append(dict(name=row["name"], turn=start, label=baseline.LABELS[label_index],\n                                              proposed_state=["positive","negative","unknown"][state],\n                                              reviewed_state="", text=" | ".join(window)))\n    report = dict(rows=len(rows), truncated_calls=int(truncated),\n                  token_quantiles={str(q):float(np.quantile(lengths,q)) for q in (.5,.9,.95,.99)},\n                  false_negative_lexical_location=counters,\n                  training_weak_state_counts={label:weak_counts[i].tolist() for i,label in enumerate(baseline.LABELS)},\n                  state_order=["positive","negative","unknown"],\n                  caveat="Rules are weak supervision, not manual evidence labels. No holdout states are used for training.")\n    exp.write_json(out/"coverage.json",report)\n    with (out/"training_state_audit.csv").open("w",encoding="utf-8-sig",newline="") as stream:\n        writer=csv.DictWriter(stream,fieldnames=["name","turn","label","proposed_state","reviewed_state","text"])\n        writer.writeheader(); writer.writerows(audit)\n    return report\n\n\ndef make_base(payload, device):\n    model = baseline._build(payload["files"], payload)\n    model.load_state_dict(payload["model"], strict=True)\n    return model.to(device)\n\n\ndef train_base(payload, tokenizer, rows, train, holdout, weights, kind, seed, out, signature, budget, device):\n    spec=dict(signature=signature, kind=kind, seed=seed, epochs=1, training_indices_sha256=exp.digest(train),\n              holdout_indices_sha256=exp.digest(holdout))\n    out=Path(out); out.mkdir(parents=True,exist_ok=True)\n    if (out/"result.json").exists():\n        if json.loads((out/"spec.json").read_text()) != spec:\n            raise ValueError("Baseline resume configuration changed")\n        return np.load(out/"holdout.npz",allow_pickle=False)["probs"].copy() if holdout else None\n    exp.write_json(out/"spec.json",spec); exp.seed_all(seed)\n    model=make_base(payload,device).train()\n    optimizer=torch.optim.AdamW(model.parameters(),lr=RECIPE["encoder_lr"],weight_decay=RECIPE["weight_decay"])\n    from transformers import get_linear_schedule_with_warmup\n    updates=exp.math_ceil(len(train)/RECIPE["effective_batch"])\n    scheduler=get_linear_schedule_with_warmup(optimizer,exp.math_ceil(updates*.1),updates)\n    scaler=torch.amp.GradScaler("cuda",enabled=device.type=="cuda")\n    order=np.random.default_rng(seed).permutation(train).tolist()\n    for start in range(0,len(order),RECIPE["effective_batch"]):\n        budget.check(); group=order[start:start+RECIPE["effective_batch"]]\n        optimizer.zero_grad(set_to_none=True)\n        for offset in range(0,len(group),RECIPE["base_micro_batch"]):\n            chunk=group[offset:offset+RECIPE["base_micro_batch"]]\n            packed,_=evidence.pack(tokenizer,[rows[i]["doc"] for i in chunk],"front",evidence.DEFAULTS)\n            packed=evidence.to_device(packed,device)\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=="cuda"):\n                logits=model(packed["ids"],packed["attention_mask"])\n            gold=torch.tensor([rows[i]["gold"] for i in chunk],device=device,dtype=torch.float32)\n            scaler.scale(objective(logits,gold,weights,kind)*len(chunk)/len(group)).backward()\n        previous=scaler.get_scale(); scaler.step(optimizer); scaler.update()\n        if scaler.get_scale()>=previous:\n            scheduler.step()\n        if start%(RECIPE["effective_batch"]*100)==0:\n            print(f"base_{kind} seed={seed} {start}/{len(train)}",flush=True)\n    model.eval()\n    texts=[baseline.transcript(rows[i]["doc"]) for i in holdout]\n    probs=baseline.probabilities(model,tokenizer,texts,batch_size=8) if holdout else None\n    export=dict(payload,model={key:value.detach().cpu() for key,value in model.state_dict().items()},\n                source=dict(experimental=True,qualified_for_submission=False,signature=signature,kind=kind,seed=seed,\n                            training_rows=len(train),official_validation_used=False))\n    exp.save_torch(out/"experimental.pt",export)\n    if holdout:\n        np.savez_compressed(out/"holdout.npz",names=[rows[i]["name"] for i in holdout],\n                            gold=[rows[i]["gold"] for i in holdout],probs=probs)\n        result=exp.metrics(np.asarray([rows[i]["gold"] for i in holdout]),probs,[rows[i]["doc"] for i in holdout])\n    else:result=dict(status="full_training_complete",training_rows=len(train))\n    exp.write_json(out/"result.json",result)\n    del model, export\n    if device.type=="cuda": torch.cuda.empty_cache()\n    return probs\n\n\ndef feature_cache(payload, tokenizer, rows, indices, out, signature, budget, device):\n    """Cache all views on the same encoder; no holdout state labels enter fitting."""\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    specification=dict(signature=signature,indices=indices,encoder_sha256=exp.digest(payload.get("source",{})))\n    marker=out/"index.json"\n    if marker.exists() and json.loads(marker.read_text()) != specification:\n        raise ValueError("Feature cache source changed")\n    exp.write_json(marker,specification)\n    model=make_base(payload,device).eval(); records={}\n    for start in range(0,len(indices),64):\n        budget.check(); chunk=indices[start:start+64]; path=out/f"shard_{start:06d}.pt"\n        if path.exists():\n            records.update(torch.load(path,map_location="cpu",weights_only=True)); continue\n        shard={}\n        for index in chunk:\n            item={}\n            for mode in ("multi","head_tail"):\n                packed,views=evidence.pack(tokenizer,[rows[index]["doc"]],mode,evidence.DEFAULTS)\n                packed=evidence.to_device(packed,device)\n                with torch.inference_mode(),torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=="cuda"):\n                    hidden=model.bert(input_ids=packed["ids"],attention_mask=packed["attention_mask"]).last_hidden_state[:,0]\n                    logits=model.classifier(hidden)\n                item[mode]=dict(cls=hidden.half().cpu(),logits=logits.float().cpu(),\n                                states=torch.tensor(data.window_states(rows[index]["doc"],views[0],data.STATE_KEYWORDS)))\n            shard[index]=item\n        exp.save_torch(path,shard); records.update(shard)\n        print(f"window features {start+len(chunk)}/{len(indices)}",flush=True)\n    del model\n    if device.type=="cuda":torch.cuda.empty_cache()\n    return records\n\n\ndef feature_batch(records, indices, mode, device):\n    selected=[records[i]["head_tail" if mode=="head_tail" else "multi"] for i in indices]\n    width=1 if mode=="front" else max(len(item["cls"]) for item in selected)\n    hidden=selected[0]["cls"].shape[-1]\n    cls=torch.zeros(len(indices),width,hidden,device=device)\n    logits=torch.zeros(len(indices),width,9,device=device)\n    mask=torch.zeros(len(indices),width,dtype=torch.bool,device=device)\n    states=torch.full((len(indices),width,9),data.IGNORE,dtype=torch.long,device=device)\n    for j,item in enumerate(selected):\n        n=min(width,len(item["cls"]))\n        cls[j,:n]=item["cls"][:n].to(device);logits[j,:n]=item["logits"][:n].to(device)\n        mask[j,:n]=True;states[j,:n]=item["states"][:n].to(device)\n    return cls,logits,mask,states\n\n\ndef head_probabilities(head,records,indices,mode,device):\n    outputs=[];head.eval()\n    with torch.inference_mode():\n        for start in range(0,len(indices),128):\n            cls,logits,mask,_=feature_batch(records,indices[start:start+128],mode,device)\n            predicted,_=head(cls,logits,mask); outputs.append(predicted.sigmoid().cpu().numpy())\n    return np.concatenate(outputs)\n\n\ndef train_head(records,rows,train,holdout,mode,seed,out,signature,budget,device):\n    specification=dict(signature=signature,mode=mode,seed=seed,epochs=RECIPE["head_epochs"],\n                       training_indices_sha256=exp.digest(train),holdout_indices_sha256=exp.digest(holdout))\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);exp.seed_all(seed)\n    head=evidence.EvidenceHead(records[train[0]]["multi"]["cls"].shape[-1],mode,evidence.DEFAULTS).to(device)\n    optimizer=torch.optim.AdamW(head.parameters(),lr=RECIPE["head_lr"],weight_decay=RECIPE["weight_decay"])\n    weights=exp.positive_weights(rows,train,device); history=[];begin=0\n    if (out/"head.pt").exists():\n        saved=torch.load(out/"head.pt",map_location=device,weights_only=True)\n        if saved["specification"]!=specification:raise ValueError("Head resume configuration changed")\n        head.load_state_dict(saved["head"]);optimizer.load_state_dict(saved["optimizer"])\n        begin=saved["epoch"]; history=saved["history"]\n    for epoch in range(begin,RECIPE["head_epochs"]):\n        exp.seed_all(seed+epoch);head.train(); order=np.random.default_rng(seed+epoch).permutation(train).tolist(); total=0\n        for start in range(0,len(order),RECIPE["head_batch"]):\n            budget.check(); chunk=order[start:start+RECIPE["head_batch"]]\n            cls,logits,mask,states=feature_batch(records,chunk,mode,device)\n            predicted,state_logits=head(cls,logits,mask)\n            gold=torch.tensor([rows[i]["gold"] for i in chunk],device=device,dtype=torch.float32)\n            value=objective(predicted,gold,weights,"bce")\n            if mode in ("states","states_removed"):\n                value=value+RECIPE["state_weight"]*auxiliary_loss(state_logits,states)\n            if not torch.isfinite(value):raise RuntimeError("Nonfinite head loss")\n            optimizer.zero_grad(set_to_none=True);value.backward();optimizer.step();total+=value.item()*len(chunk)\n        record=dict(epoch=epoch+1,loss=total/len(train));history.append(record)\n        exp.save_torch(out/"head.pt",dict(specification=specification,head=head.state_dict(),optimizer=optimizer.state_dict(),\n                                         epoch=epoch+1,history=history))\n        print(f"{mode} seed={seed} head epoch={epoch+1} loss={record[\'loss\']:.4f}",flush=True)\n    probs=head_probabilities(head,records,holdout,mode,device) if holdout else None\n    if probs is not None:\n        np.savez_compressed(out/"holdout.npz",names=[rows[i]["name"] for i in holdout],\n                            gold=[rows[i]["gold"] for i in holdout],probs=probs)\n        exp.write_json(out/"result.json",exp.metrics(np.asarray([rows[i]["gold"] for i in holdout]),probs,\n                                                    [rows[i]["doc"] for i in holdout]))\n    exp.write_json(out/"history.json",history)\n    return head,probs\n\n\ndef fine_model(payload,tokenizer,rows,train,holdout,mode,seed,head,out,signature,budget,device):\n    """One encoder epoch after a fixed frozen-head fit; final epoch only."""\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    spec=dict(signature=signature,mode=mode,seed=seed,train=exp.digest(train),holdout=exp.digest(holdout))\n    if (out/"experimental.pt").exists():\n        saved=torch.load(out/"experimental.pt",map_location="cpu",weights_only=True)\n        if saved["source"]["specification"]!=spec:raise ValueError("Fine resume configuration changed")\n        return saved,np.load(out/"holdout.npz")["probs"].copy() if holdout else None\n    exp.seed_all(seed)\n    model=evidence.EvidenceModel(make_base(payload,device),mode).to(device)\n    model.head.load_state_dict(head.state_dict());model.train()\n    optimizer=torch.optim.AdamW([\n        dict(params=model.base.parameters(),lr=RECIPE["encoder_lr"]),\n        dict(params=model.head.parameters(),lr=RECIPE["head_fine_lr"])],weight_decay=RECIPE["weight_decay"])\n    weights=exp.positive_weights(rows,train,device)\n    scaler=torch.amp.GradScaler("cuda",enabled=device.type=="cuda")\n    from transformers import get_linear_schedule_with_warmup\n    updates=exp.math_ceil(len(train)/RECIPE["effective_batch"])\n    scheduler=get_linear_schedule_with_warmup(optimizer,exp.math_ceil(updates*.1),updates)\n    order=np.random.default_rng(seed).permutation(train).tolist()\n    for start in range(0,len(order),RECIPE["effective_batch"]):\n        budget.check();group=order[start:start+RECIPE["effective_batch"]];optimizer.zero_grad(set_to_none=True)\n        for offset in range(0,len(group),RECIPE["window_micro_batch"]):\n            chunk=group[offset:offset+RECIPE["window_micro_batch"]]\n            documents=[rows[i]["doc"] for i in chunk]\n            packed,views=evidence.pack(tokenizer,documents,mode,model.config)\n            states=torch.full((len(chunk),packed["width"],9),data.IGNORE,device=device,dtype=torch.long)\n            for j,(doc,windows) in enumerate(zip(documents,views)):\n                states[j,:len(windows)]=torch.tensor(data.window_states(doc,windows,data.STATE_KEYWORDS),device=device)\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=="cuda"):\n                logits,state_logits=model(evidence.to_device(packed,device))\n            gold=torch.tensor([rows[i]["gold"] for i in chunk],device=device,dtype=torch.float32)\n            value=objective(logits,gold,weights,"bce")\n            if mode in ("states","states_removed"):value+=RECIPE["state_weight"]*auxiliary_loss(state_logits,states)\n            if not torch.isfinite(value):raise RuntimeError("Nonfinite fine loss")\n            scaler.scale(value*len(chunk)/len(group)).backward()\n        previous=scaler.get_scale();scaler.step(optimizer);scaler.update()\n        if scaler.get_scale()>=previous:scheduler.step()\n        if start%(RECIPE["effective_batch"]*100)==0:print(f"fine {mode} seed={seed} {start}/{len(train)}",flush=True)\n    model.eval()\n    probs=evidence.probabilities(model,tokenizer,[rows[i]["doc"] for i in holdout]) if holdout else None\n    if holdout:\n        np.savez_compressed(out/"holdout.npz",names=[rows[i]["name"] for i in holdout],gold=[rows[i]["gold"] for i in holdout],probs=probs)\n        exp.write_json(out/"result.json",exp.metrics(np.asarray([rows[i]["gold"] for i in holdout]),probs))\n    exported=evidence.export_payload(model,payload,dict(experimental=True,qualified_for_submission=False,specification=spec))\n    exp.save_torch(out/"experimental.pt",exported)\n    del model\n    if device.type=="cuda":torch.cuda.empty_cache()\n    return exported,probs\n\n\ndef checked_gate(gold,candidate,ensemble,control,ablation,rows,holdout):\n    gate=exp.adoption_gate(gold,candidate,ensemble,control,ablation,calls=[rows[i]["call"] for i in holdout],smoke=False)\n    gate["checks"].update(gate["contribution_checks"])\n    gate["passed"]=all(gate["checks"].values())\n    return gate\n\n\ndef run(args):\n    started=time.monotonic();out=args.out;out.mkdir(parents=True,exist_ok=True)\n    RECIPE["head_epochs"]=2 if args.smoke_rows else 10\n    budget=exp.RunBudget(out,args.max_hours);device=torch.device(args.device);torch.set_num_threads(4)\n    if device.type=="cuda" and not torch.cuda.is_available():raise RuntimeError("CUDA GPU required")\n    rows,train,holdout,base_probs,ensemble=exp.prepare_data(args.train_label_dir,args.reference_probs_dir,args.smoke_rows)\n    source,signature=source_signature(rows,train,holdout,args.base_ckpt,args.smoke_rows,device)\n    marker=out/"source.json"\n    if marker.exists() and json.loads(marker.read_text(encoding="utf-8"))!=source:\n        raise ValueError("Experiment resume source/configuration changed")\n    exp.write_json(marker,source)\n    payload=torch.load(args.base_ckpt,map_location="cpu",weights_only=True)\n    if payload.get("format")!=baseline.FORMAT or tuple(payload.get("labels",[]))!=baseline.LABELS:\n        raise ValueError("A self-contained RoBERTa baseline is required")\n    if not args.smoke_rows and (payload.get("source",{}).get("exp_id")!="M3-ROBERTA-003" or payload["source"].get("seed")!=42):\n        raise ValueError("ROBERTA-003 seed42 required")\n    from transformers import AutoTokenizer\n    import tempfile\n    with tempfile.TemporaryDirectory() as directory:\n        for name,value in payload["files"].items():\n            if Path(name).name!=name:raise ValueError("Unsafe tokenizer file")\n            (Path(directory)/name).write_text(value,encoding="utf-8")\n        tokenizer=AutoTokenizer.from_pretrained(directory,local_files_only=True,use_fast=True)\n    summary=dict(status="running",smoke=bool(args.smoke_rows),source=source,selected_method=None,\n                 official_validation_used=False,pilot={},loss_comparison={},gate={"passed":False})\n    gold=np.asarray([rows[i]["gold"] for i in holdout]);docs=[rows[i]["doc"] for i in holdout]\n    summary["pilot"].update(roberta003=exp.metrics(gold,base_probs,docs),existing_ensemble=exp.metrics(gold,ensemble,docs))\n    def save():\n        summary["elapsed_seconds"]=time.monotonic()-started;exp.report(out,summary)\n        report=out/"report.md"\n        text=report.read_text(encoding="utf-8").replace("# M3 context experiment","# M3 loss, coverage and evidence-state experiment",1)\n        text=text.replace("Training-only retrieval.","Training-only weak state supervision; at most six windows.")\n        if summary["loss_comparison"]:\n            text+="\\n## Matched one-epoch loss comparison\\n\\n| Loss | Macro F1 | Micro F1 |\\n|---|---:|---:|\\n"\n            for name,item in summary["loss_comparison"].items():text+=f"| {name} | {item[\'macro_f1\']:.4f} | {item[\'micro_f1\']:.4f} |\\n"\n        report.write_text(text,encoding="utf-8")\n    try:\n        inspect_coverage(rows,train,holdout,tokenizer,base_probs,out,budget)\n        save()\n        if args.phase=="audit":summary["status"]="audit_complete";save();return summary\n        weights=exp.positive_weights(rows,train,device)\n        for kind in ("original","bce","asl"):\n            probs=train_base(payload,tokenizer,rows,train,holdout,weights,kind,42,out/f"loss_{kind}_42",signature,budget,device)\n            summary["loss_comparison"][kind]=exp.metrics(gold,probs,docs);save()\n        # Loss selection is a screening decision, not a submission qualification.\n        loss_winner=max(("bce","asl"),key=lambda name:summary["loss_comparison"][name]["macro_f1"])\n        minimum=max(summary["loss_comparison"]["original"]["macro_f1"],summary["pilot"]["roberta003"]["macro_f1"])\n        if summary["loss_comparison"][loss_winner]["macro_f1"]>minimum:\n            chosen=torch.load(out/f"loss_{loss_winner}_42/experimental.pt",map_location="cpu",weights_only=True)\n            del payload;payload=chosen;summary["feature_encoder_loss"]=loss_winner\n        else:summary["feature_encoder_loss"]="original_checkpoint"\n        if args.phase=="loss":summary["status"]="loss_screen_complete";save();return summary\n        records=feature_cache(payload,tokenizer,rows,sorted(set(train+holdout)),out/"features",signature,budget,device)\n        methods=("front","head_tail","multi","states","states_removed")\n        for mode in methods:\n            _,probs=train_head(records,rows,train,holdout,mode,42,out/f"{mode}_42",signature,budget,device)\n            summary["pilot"][mode]=exp.metrics(gold,probs,docs);save()\n        floor=max(summary["pilot"][name]["macro_f1"] for name in ("front","roberta003","existing_ensemble"))\n        if summary["feature_encoder_loss"]!="original_checkpoint":\n            floor=max(floor,summary["loss_comparison"][loss_winner]["macro_f1"])\n        successful=[name for name in ("head_tail","multi","states") if summary["pilot"][name]["macro_f1"]>floor]\n        winner=max(successful,key=lambda name:summary["pilot"][name]["macro_f1"]) if successful else None\n        summary.update(status="smoke_complete" if args.smoke_rows else "input_pilots_failed" if winner is None else "repeat_pending",selected_method=winner)\n        save()\n        if args.smoke_rows or args.phase=="pilot":return summary\n        # A loss-only winner is useful even when richer inputs do not help.\n        if winner is None:\n            if summary["loss_comparison"][loss_winner]["macro_f1"]<=max(minimum,summary["pilot"]["existing_ensemble"]["macro_f1"]):return summary\n            candidates=[];controls=[]\n            for seed in RECIPE["seeds"]:\n                candidates.append(train_base(torch.load(args.base_ckpt,map_location="cpu",weights_only=True),tokenizer,rows,train,holdout,weights,loss_winner,seed,out/f"loss_{loss_winner}_{seed}",signature,budget,device))\n                controls.append(train_base(torch.load(args.base_ckpt,map_location="cpu",weights_only=True),tokenizer,rows,train,holdout,weights,"original",seed,out/f"loss_original_{seed}",signature,budget,device))\n            summary["gate"]=checked_gate(gold,candidates,ensemble,controls,controls,rows,holdout)\n            summary["selected_method"]="loss_"+loss_winner\n            summary["status"]="repeat_gate_failed"\n            if summary["gate"]["passed"]:\n                full=list(range(len(rows)))\n                original=torch.load(args.base_ckpt,map_location="cpu",weights_only=True)\n                train_base(original,tokenizer,rows,full,[],exp.positive_weights(rows,full,device),loss_winner,42,out/"full_loss",signature,budget,device)\n                final=torch.load(out/"full_loss/experimental.pt",map_location="cpu",weights_only=True)\n                final["source"].update(smoke=False,training_rows=len(full),representative_seed=42,internal_gate=summary["gate"],official_validation_used=False,qualified_for_submission=False)\n                exp.save_torch(out/"final_candidate.pt",final);summary["status"]="final_candidate_ready"\n            save();return summary\n        candidate,control,ablation=[],[],[]\n        ablated="states_removed" if winner=="states" else "front"\n        summary["repeat"]={winner:{"frozen":[]}}\n        for seed in RECIPE["seeds"]:\n            for mode,destination in ((winner,candidate),("front",control),(ablated,ablation)):\n                _,probs=train_head(records,rows,train,holdout,mode,seed,out/f"{mode}_{seed}",signature,budget,device)\n                destination.append(probs)\n                if mode==winner:summary["repeat"][winner]["frozen"].append(exp.metrics(gold,probs,docs))\n            save()\n        summary["frozen_gate"]=checked_gate(gold,candidate,ensemble,control,ablation,rows,holdout)\n        summary["gate"]=summary["frozen_gate"]\n        fine_candidate=[];fine_control=[];fine_ablation=[]\n        summary["repeat"][winner]["fine"]=[]\n        for seed in RECIPE["seeds"]:\n            for mode,destination in ((winner,fine_candidate),("front",fine_control),(ablated,fine_ablation)):\n                head,_=train_head(records,rows,train,holdout,mode,seed,out/f"{mode}_{seed}",signature,budget,device)\n                _,probs=fine_model(payload,tokenizer,rows,train,holdout,mode,seed,head,out/f"fine_{mode}_{seed}",signature,budget,device)\n                destination.append(probs)\n                if mode==winner:summary["repeat"][winner]["fine"].append(exp.metrics(gold,probs,docs))\n            save()\n        summary["fine_gate"]=checked_gate(gold,fine_candidate,ensemble,fine_control,fine_ablation,rows,holdout)\n        # Fix the recipe using internal results before any official access.\n        stage="fine" if summary["fine_gate"]["passed"] and (not summary["frozen_gate"]["passed"] or summary["fine_gate"]["mean_macro_f1"]>summary["frozen_gate"]["mean_macro_f1"]) else "frozen"\n        summary["gate"]=summary[stage+"_gate"];summary["selected_stage"]=stage\n        summary["status"]="repeat_gate_failed"\n        if summary["gate"]["passed"]:\n            full=list(range(len(rows)))\n            # Recreate the chosen loss adaptation on all Training, not the internal split.\n            if summary["feature_encoder_loss"]!="original_checkpoint":\n                original=torch.load(args.base_ckpt,map_location="cpu",weights_only=True)\n                train_base(original,tokenizer,rows,full,[],exp.positive_weights(rows,full,device),loss_winner,42,out/"full_loss",signature,budget,device)\n                payload=torch.load(out/"full_loss/experimental.pt",map_location="cpu",weights_only=True)\n                records=feature_cache(payload,tokenizer,rows,full,out/"full_features",signature,budget,device)\n            head,_=train_head(records,rows,full,[],winner,42,out/"full_head",signature,budget,device)\n            if stage=="fine":final,_=fine_model(payload,tokenizer,rows,full,[],winner,42,head,out/"full_fine",signature,budget,device)\n            else:\n                model=evidence.EvidenceModel(make_base(payload,device),winner).to(device);model.head.load_state_dict(head.state_dict())\n                final=evidence.export_payload(model,payload,{})\n            final["source"].update(smoke=False,training_rows=len(full),representative_seed=42,internal_gate=summary["gate"],signature=signature,official_validation_used=False,selected_stage=stage,qualified_for_submission=False,experimental=True)\n            exp.save_torch(out/"final_candidate.pt",final);summary["status"]="final_candidate_ready"\n        save();return summary\n    except exp.BudgetReached:\n        summary["status"]="budget_reached_resume_pending";save();return summary\n    finally:\n        try:budget.check(force=True)\n        except exp.BudgetReached:pass\n', 'src/dcc_inference/mission3_evidence.py': '"""Offline symptom evidence model with bounded utterance windows and state head."""\nfrom __future__ import annotations\nimport json\nimport math\nimport tempfile\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\ntry:\n    from . import mission3_bert as baseline\nexcept ImportError:\n    import mission3_bert as baseline\ntry:\n    from dcc_data.mission3_evidence_data import tokenize_views\nexcept ImportError:\n    from mission3_evidence_data import tokenize_views\n\nFORMAT = "dcc-m3-evidence-v2"\nDEFAULTS = dict(dimension=128, dropout=.1, max_length=256, max_windows=6)\n\n\nclass EvidenceHead(nn.Module):\n    def __init__(self, hidden, mode, config):\n        super().__init__()\n        self.mode = mode\n        dimension = config["dimension"]\n        self.project = nn.Sequential(nn.Linear(hidden, dimension), nn.Tanh(), nn.Dropout(config["dropout"]))\n        self.queries = nn.Parameter(torch.randn(9, dimension)/math.sqrt(dimension))\n        self.states = nn.Linear(dimension, 9*3)\n        self.residual = nn.Linear(dimension+3, 1)\n        nn.init.zeros_(self.residual.weight); nn.init.zeros_(self.residual.bias)\n\n    def forward(self, cls, logits, mask):\n        hidden = self.project(cls.float())\n        state_logits = self.states(hidden).reshape(*hidden.shape[:2], 9, 3)\n        attention = torch.einsum("bwd,ld->blw", hidden, self.queries)/math.sqrt(hidden.shape[-1])\n        attention = attention.masked_fill(~mask[:, None], -1e4).softmax(-1)\n        pooled = torch.einsum("blw,bwd->bld", attention, hidden)\n        state_probs = torch.einsum("blw,bwls->bls", attention, state_logits.softmax(-1))\n        if self.mode not in ("states", "states_removed"):\n            state_probs = state_probs*0\n        elif self.mode == "states_removed":\n            state_probs = state_probs*0\n        correction = self.residual(torch.cat([pooled, state_probs], -1)).squeeze(-1)\n        return logits[:, 0].float()+correction, state_logits\n\n\nclass EvidenceModel(nn.Module):\n    requires_documents = True\n    checkpoint_format = FORMAT\n\n    def __init__(self, base, mode, config=None):\n        super().__init__()\n        self.base, self.mode = base, mode\n        self.config = dict(DEFAULTS, **(config or {}))\n        self.head = EvidenceHead(base.classifier.in_features, mode, self.config)\n\n    def extract(self, ids, attention_mask, owners, window_numbers, count, width):\n        hidden = self.base.bert(input_ids=ids, attention_mask=attention_mask).last_hidden_state[:, 0]\n        logits = self.base.classifier(self.base.dropout(hidden))\n        cls = hidden.new_zeros(count, width, hidden.shape[-1], dtype=torch.float32)\n        scores = logits.new_zeros(count, width, 9, dtype=torch.float32)\n        mask = torch.zeros(count, width, dtype=torch.bool, device=ids.device)\n        cls[owners, window_numbers] = hidden.float()\n        scores[owners, window_numbers] = logits.float()\n        mask[owners, window_numbers] = True\n        return cls, scores, mask\n\n    def forward(self, packed):\n        cls, logits, mask = self.extract(**packed)\n        with torch.autocast(device_type=cls.device.type, enabled=False):\n            return self.head(cls, logits, mask)\n\n\ndef pack(tokenizer, documents, mode, config):\n    views = [tokenize_views(tokenizer, doc, mode, config["max_length"], config["max_windows"]) for doc in documents]\n    flat = [dict(input_ids=v["ids"], attention_mask=[1]*len(v["ids"])) for windows in views for v in windows]\n    encoded = tokenizer.pad(flat, padding=True, return_tensors="pt")\n    packed = dict(ids=encoded["input_ids"], attention_mask=encoded["attention_mask"],\n                  owners=torch.tensor([i for i, windows in enumerate(views) for _ in windows]),\n                  window_numbers=torch.tensor([j for windows in views for j in range(len(windows))]),\n                  count=len(documents), width=max(map(len, views)))\n    return packed, views\n\n\ndef to_device(packed, device):\n    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in packed.items()}\n\n\ndef export_payload(model, original, source):\n    return dict(format=FORMAT, files=original["files"], labels=list(baseline.LABELS),\n                max_length=model.config["max_length"], config=model.config, mode=model.mode,\n                model={key: value.detach().cpu() for key, value in model.state_dict().items()}, source=source)\n\n\ndef load_payload(payload, device=None):\n    from transformers import AutoTokenizer\n    if payload.get("format") != FORMAT or tuple(payload.get("labels", [])) != baseline.LABELS:\n        raise ValueError("Invalid evidence checkpoint")\n    files = payload["files"]\n    with tempfile.TemporaryDirectory() as folder:\n        for name, value in files.items():\n            if Path(name).name != name:\n                raise ValueError("Unsafe tokenizer file name")\n            (Path(folder)/name).write_text(value, encoding="utf-8")\n        tokenizer = AutoTokenizer.from_pretrained(folder, local_files_only=True, use_fast=True)\n    base = baseline._build(files, payload)\n    model = EvidenceModel(base, payload["mode"], payload["config"])\n    model.load_state_dict(payload["model"], strict=True)\n    model.to(device or ("cuda" if torch.cuda.is_available() else "cpu")).eval()\n    return model, tokenizer, list(baseline.LABELS), model.config["max_length"]\n\n\ndef probabilities(model, tokenizer, documents, names=None, max_length=256, batch_size=2):\n    if max_length != model.config["max_length"]:\n        raise ValueError("Inference length must match the checkpoint")\n    device = next(model.parameters()).device\n    out = np.zeros((len(documents), 9), dtype=np.float32)\n    with torch.inference_mode():\n        for start in range(0, len(documents), batch_size):\n            packed, _ = pack(tokenizer, documents[start:start+batch_size], model.mode, model.config)\n            logits, _ = model(to_device(packed, device))\n            out[start:start+batch_size] = logits.float().sigmoid().cpu().numpy()\n    return out\n', 'scripts/train_mission3_next.py': '"""Run M3 controlled loss and complete-window evidence pilots on Training only."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/"src"))\nfrom dcc_data.mission3_next_experiment import run\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser(description=__doc__)\n    for option in ("train-label-dir","base-ckpt","reference-probs-dir","out"):\n        parser.add_argument("--"+option,type=Path,required=True)\n    parser.add_argument("--device",choices=("cpu","cuda"),default="cuda")\n    parser.add_argument("--phase",choices=("audit","loss","pilot","all"),default="all")\n    parser.add_argument("--smoke-rows",type=int,default=0)\n    parser.add_argument("--max-hours",type=float,default=6.)\n    args=parser.parse_args()\n    if args.smoke_rows<0 or not 0<args.max_hours<=10:parser.error("Invalid smoke rows/budget")\n    result=run(args)\n    print(json.dumps({key:result[key] for key in ("status","selected_method","elapsed_seconds")},ensure_ascii=False))\n', 'scripts/evaluate_mission3_context.py': '"""Use official Validation once for the internally accepted final candidate."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nROOT=Path(__file__).resolve().parents[1]\nsys.path.insert(0,str(ROOT/"src"))\nfrom dcc_data.mission3_context_release import evaluate_official\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--ckpt",required=True,type=Path)\n    parser.add_argument("--official-label-dir",required=True,type=Path)\n    parser.add_argument("--out",required=True,type=Path)\n    parser.add_argument("--registry",type=Path,default=ROOT/"reports/mission3/context/official_once.json")\n    parser.add_argument("--device",choices=("cpu","cuda"))\n    parser.add_argument("--batch",type=int,default=32)\n    args=parser.parse_args()\n    if args.batch<1:\n        parser.error("batch must be positive")\n    result=evaluate_official(args.ckpt,args.official_label_dir,args.out,args.registry,args.device,args.batch)\n    print(json.dumps(dict(macro_f1=result["metrics"]["macro_f1"],decision=result["decision"])))\n\n\nif __name__=="__main__":\n    main()\n', 'scripts/package_mission3_context.py': '"""Freeze a separately named offline package only after official acceptance."""\nimport argparse\nimport sys\nfrom pathlib import Path\n\nROOT=Path(__file__).resolve().parents[1]\nsys.path.insert(0,str(ROOT/"src"))\nfrom dcc_data.mission3_context_release import package_candidate\n\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--ckpt",required=True,type=Path)\n    parser.add_argument("--official-result",required=True,type=Path)\n    parser.add_argument("--out",required=True,type=Path)\n    args=parser.parse_args()\n    print(package_candidate(args.ckpt,args.official_result,args.out,ROOT))\n', 'submission/mission3_evidence/inference.py': '"""Offline M3 inference, preserving the existing CLI and CSV schema."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport pandas as pd\nimport mission3_bert as baseline\nimport mission3_evidence as evidence\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    for option in ("audio_dir","label_dir","ckpt_path"):\n        parser.add_argument("--"+option,type=Path,required=True)\n    parser.add_argument("--output",type=Path,default=Path("outputs/mission3.csv"))\n    args=parser.parse_args()\n    paths=sorted(args.label_dir.rglob("*.json"))\n    if not paths or len({p.name for p in paths})!=len(paths):raise ValueError("Expected uniquely named JSON files")\n    docs=[json.loads(path.read_text(encoding="utf-8-sig")) for path in paths]\n    model,tokenizer,labels,length=baseline.load_checkpoint(args.ckpt_path)\n    if getattr(model,"requires_documents",False):\n        probs=evidence.probabilities(model,tokenizer,docs,max_length=length,batch_size=2)\n    else:probs=baseline.probabilities(model,tokenizer,[baseline.transcript(doc) for doc in docs],length,8)\n    frame=pd.DataFrame({"label file name":[path.name for path in paths],"symptom":[str([label for label,positive in zip(labels,row) if positive]) for row in probs>=.5]})\n    args.output.parent.mkdir(parents=True,exist_ok=True)\n    frame.to_csv(args.output,index=False,encoding="utf-8")\n    print(f"Mission 3 CSV saved: {args.output} (rows={len(frame)})")\n\n\nif __name__=="__main__":main()\n', 'submission/mission3_evidence/requirements.txt': 'torch==2.12.1\ntransformers==4.57.6\nnumpy\npandas\nsafetensors\n', 'submission/mission3_evidence/README.md': '# M3 offline candidate\n\nThis is a packaging template, not an adopted model. `mission3.pt` is added only\nafter three-seed internal qualification and official Macro F1 >= 0.6540.\n\nRun offline:\n```\npython inference.py --audio_dir AUDIO --label_dir LABELS --ckpt_path mission3.pt --output outputs/mission3.csv\n```\nOnly `utterances[].text` is read. Nine labels keep their original order and\nthreshold 0.5. The checkpoint contains model weights, tokenizer, module settings\nand provenance; no model download is required. At most six 256-token windows are\nused, so very long calls may have gaps between sampled windows.\n'}
MANIFEST={'src/dcc_data/__init__.py': 'e9eb24d361066281dfa5fd719ad5f7beb75be5d4f65ded7a8545873e57288492', 'src/dcc_data/manifest.py': 'fa3ddf20437d71119c123a1183356b3ed00d79958094f7bc746e74fe965e26e2', 'src/dcc_data/guards.py': 'f63a933b45f3619eeefdc1ac595c278bbc7bd1b24564b1d0388b5323eca0f300', 'src/dcc_inference/__init__.py': '7548e16df78d2bd4a8b2e6c23d049cb6f844c47dd7f0c38b726f2e854bb7f8a2', 'src/dcc_data/mission3_context_experiment.py': 'f52857b6b856914217d88c29181f4ee2b9f0aafc21af596d086c07f5608b126d', 'src/dcc_data/mission3_context_release.py': '49f67e213949676e99ad75180719eb9c8492019721ec4826281709da72cd435a', 'src/dcc_inference/mission3_bert.py': 'd426d3c6add0b9e8bf01438e07edb14de750444f95419d8124cc325bc5895a99', 'src/dcc_inference/mission3_context.py': '08f3d83816c5c398e40b831c0f289f8efce012d4dbf9259d5e25f6cf6c84ce12', 'src/dcc_data/mission3_evidence_data.py': '3f3af37e9e643acdf34636de02081caec22c3f73701cd3797c1aec3d20e7bfdf', 'src/dcc_data/mission3_next_experiment.py': 'a6d2a2b668e31b8ae71ab9fc0856839148030bc803fb2329dea7df35d9abe20d', 'src/dcc_inference/mission3_evidence.py': '00a56fe10c848f0201fd2ec30d1ecdce17f6552a5f26d51c6405abc42234c155', 'scripts/train_mission3_next.py': '93d87974edb070609c4cbda2d2c60a4e00b49482ac6e5d1156936104b04f8c7f', 'scripts/evaluate_mission3_context.py': 'e86537a963597bd3a5ff6b9d39134778a5e0afefd5d941c5a80002456719802b', 'scripts/package_mission3_context.py': 'ca241c651db9bfd02b91bbcc8b752b278067d727d7d8eaeac3ebdde47e55c8e4', 'submission/mission3_evidence/inference.py': 'bb5d44b6a14d28e9ed903b7979e1aceb90306566d7a86332712ef5f37e2d4c5a', 'submission/mission3_evidence/requirements.txt': '5bba47e5b58793c6164bbf548a32e4471df6a7b6acb6443f70abfd50ec327bfa', 'submission/mission3_evidence/README.md': 'c415dc83bb480ecc12f16783003ee5b15946c4e35b10102ec3f87dd5575ddf78'}
for name,text in SOURCES.items():
    target=ROOT/name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(text,encoding='utf-8')
    assert hashlib.sha256(target.read_bytes()).hexdigest()==MANIFEST[name]
print('Source files verified:',len(MANIFEST))


In [ ]:
INPUT=Path('/content/input.tar.gz')
assert INPUT.is_file(), 'input.tar.gz를 세션 저장소에 업로드하세요.'
if not (ROOT/'bundle.json').exists():
    with tarfile.open(INPUT) as archive:
        archive.extractall(ROOT,filter='data')
# The old input archive also contains v1 source; restore the fixed v2 source afterwards.
for name,text in SOURCES.items():
    (ROOT/name).parent.mkdir(parents=True,exist_ok=True)
    (ROOT/name).write_text(text,encoding='utf-8')
meta=json.loads((ROOT/'bundle.json').read_text())
assert (meta['train_rows'],meta['internal_holdout_rows'],meta['training_rows'])==(23408,5792,29200)
assert not meta['official_validation_included']
def sha(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(1<<20),b''):digest.update(block)
    return digest.hexdigest()
assert sha(ROOT/'baseline.pt')==meta['checkpoint_sha256']
print('Private inputs verified:',meta)


In [ ]:
def run_next(phase,smoke=0):
    output=ROOT/('smoke' if smoke else 'run')
    output.mkdir(exist_ok=True)
    cmd=[sys.executable,'-u',str(ROOT/'scripts/train_mission3_next.py'),'--train-label-dir',str(ROOT/'training'),'--base-ckpt',str(ROOT/'baseline.pt'),'--reference-probs-dir',str(ROOT/'reference'),'--out',str(output),'--device','cuda','--phase',phase,'--smoke-rows',str(smoke),'--max-hours','0.25' if smoke else '6']
    with (output/'execution.log').open('a',encoding='utf-8') as log:
        process=subprocess.Popen(cmd,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',env=dict(os.environ,USE_TF='0'))
        for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        if process.wait():raise RuntimeError('실험 실패: execution.log 확인')
    return json.loads((output/'metrics.json').read_text())
smoke_result=run_next('all',16)
assert smoke_result['smoke'] and not smoke_result['gate']['passed']
print(smoke_result['status'])


In [ ]:
# Stage 1: same checkpoint, one encoder epoch, three loss functions.
loss_result=run_next('loss')
print(json.dumps(loss_result.get('loss_comparison',{}),ensure_ascii=False,indent=2))


In [ ]:
# Stage 2 resumes Stage 1; all input pilots, then only a successful winner.
result=run_next('all')
print('Status:',result['status'],'Selected:',result['selected_method'])
print('Internal gate:',json.dumps(result['gate'],ensure_ascii=False,indent=2))
print('Official Validation remains unused.')


## 보존 후 런타임 해제

결과 ZIP과 가중치를 각각 다운로드합니다. ZIP 무결성·파일 크기·SHA-256을 로컬에서 확인하기 전까지 런타임을 삭제하지 마세요. `experimental.pt`도 다운로드 대상으로 포함됩니다. Drive에는 자동 전송하지 않습니다. 전체 Training 후보가 생기면 `final_candidate.pt`를 먼저 확보합니다.

In [ ]:
from google.colab import files
run=ROOT/'run'
weights=sorted(run.rglob('experimental.pt'))+sorted(run.glob('final_candidate.pt'))
checkpoint_manifest={str(path.relative_to(ROOT)):dict(bytes=path.stat().st_size,sha256=sha(path)) for path in weights}
(run/'checkpoint_manifest.json').write_text(json.dumps(checkpoint_manifest,indent=2))
archive_path=Path('/content/m3_next_results.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run.rglob('*')):
        if path.is_file() and path.suffix in ('.json','.md','.log','.npz','.png','.csv'):
            archive.write(path,path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive_path) as archive:assert archive.testzip() is None
print('Results:',archive_path.stat().st_size,sha(archive_path))
print('Checkpoint manifest:',json.dumps(checkpoint_manifest,indent=2))
files.download(str(archive_path))


In [ ]:
# Download actual weights separately; they are NOT inside the metrics ZIP.
for path in weights:
    print('Downloading weights:',path.name,path.stat().st_size,sha(path))
    files.download(str(path))
print('Verify local files against checkpoint_manifest.json before deleting the runtime.')


## 조건부 공식 평가와 패키지

`final_candidate_ready`일 때만 전체 Training 후보를 로컬에 내려받습니다. 기존 `scripts/evaluate_mission3_context.py`로 공식 평가를 실행하며 공유 `official_once.json`을 유지합니다. 공식 Macro F1 ≥ 0.6540일 때만 `scripts/package_mission3_context.py`로 새 디렉터리에 패키지를 만듭니다. 새 모델을 위해 공식 평가 기록을 초기화하거나 공식 점수로 설정을 조정하지 않습니다.